<a href="https://colab.research.google.com/github/heeseok00/project_DScover_Price_Prediction/blob/main/%EA%B9%80%EC%A7%84%EC%84%B1_0328%EA%B0%80%EC%9D%B4%EB%93%9C%ED%94%84%EB%A1%9C%EC%A0%9D%ED%8A%B8_%EB%AA%A8%EB%8D%B8%EB%A7%81.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  fonts-nanum
0 upgraded, 1 newly installed, 0 to remove and 6 not upgraded.
Need to get 10.3 MB of archives.
After this operation, 34.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-nanum all 20200506-1 [10.3 MB]
Fetched 10.3 MB in 0s (30.4 MB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package fonts-nanum.
(Reading database ... 118194 files and direct

In [1]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 나눔바른고딕 폰트 경로 설정
font_path = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf'
font_name = fm.FontProperties(fname=font_path).get_name()

plt.rc('font', family=font_name)
plt.rcParams['axes.unicode_minus'] = False

print(f'설정된 폰트: {font_name}')

설정된 폰트: NanumBarunGothic


In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

# 1. 데이터 로드 및 전처리 (예시: 배추)
train = pd.read_csv('train.csv')
supply = pd.read_csv('TRAIN_산지공판장_2018-2021.csv')

In [4]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# 1. 데이터 로드
train = pd.read_csv('train.csv')
wholesale = pd.read_csv('TRAIN_전국도매_2018-2021.csv')

# 2. 사과 '후지'로 수정한 10대 타겟 품목
target_items = [
    {'품목명': '건고추', '품종명': '화건', '거래단위': '30 kg', '등급': '상품'},
    {'품목명': '사과', '품종명': '후지', '거래단위': '10 개', '등급': '상품'}, # 홍로 -> 후지로 변경
    {'품목명': '감자', '품종명': '감자 수미', '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '배', '품종명': '신고', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '깐마늘(국산)', '품종명': '깐마늘(국산)', '거래단위': '20 kg', '등급': '상품'},
    {'품목명': '무', '품종명': '무', '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '상추', '품종명': '청', '거래단위': '100 g', '등급': '상품'},
    {'품목명': '배추', '품종명': '배추', '거래단위': '10키로망대', '등급': '상'},
    {'품목명': '양파', '품종명': '양파', '거래단위': '1키로', '등급': '상'},
    {'품목명': '대파', '품종명': '대파(일반)', '거래단위': '1키로단', '등급': '상'}
]

# 전체 144개 시점 뼈대 만들기
all_time_steps = train['시점'].unique()
all_time_df = pd.DataFrame({'시점': all_time_steps}).sort_values('시점')

imputed_results = {}

for item in target_items:
    name = f"{item['품목명']}_{item['품종명']}"

    # ----------------------------------------------------
    # [Step 1] 가격 데이터(y_t) 추출
    # ----------------------------------------------------
    cond_price = (train['품목명'] == item['품목명']) & \
                 (train['품종명'] == item['품종명']) & \
                 (train['거래단위'] == item['거래단위']) & \
                 (train['등급'] == item['등급'])
    df_price = train[cond_price][['시점', '평균가격(원)']]

    # ----------------------------------------------------
    # [Step 2] 전국도매 총반입량 데이터(x_t) 추출 및 집계
    # ----------------------------------------------------
    cond_supply = (wholesale['품목명'] == item['품목명']) & \
                  (wholesale['품종명'] == item['품종명'])
    df_supply = wholesale[cond_supply].groupby('시점')['총반입량(kg)'].sum().reset_index()

    # ----------------------------------------------------
    # [Step 3] 전체 시점 뼈대에 병합
    # ----------------------------------------------------
    df_merged = pd.merge(all_time_df, df_price, on='시점', how='left')
    df_merged = pd.merge(df_merged, df_supply, on='시점', how='left')

    # 🚨 중요: 가격(endog)은 결측치여도 칼만이 알아서 채워주지만,
    # 외생변수(exog)는 결측치가 있으면 에러가 납니다. 반입량이 없는 구간은 0으로 채웁니다.
    df_merged['총반입량(kg)'] = df_merged['총반입량(kg)'].fillna(0)

    # 안전장치
    valid_count = df_merged['평균가격(원)'].notna().sum()
    if valid_count == 0:
        print(f"[{name}] 데이터 0건. 건너뜁니다.")
        continue

    # ----------------------------------------------------
    # [Step 4] 상태 공간 모델 적용 (우리의 수식 구현!)
    # ----------------------------------------------------
    try:
        model = sm.tsa.UnobservedComponents(
            endog=df_merged['평균가격(원)'],      # y_t
            level='local linear trend',         # 추세 (수준 + 기울기)
            seasonal=36,                        # 계절성 (Rotate 36순)
            autoregressive=1,                   # 잔차 전이 (AR 1)
            exog=df_merged['총반입량(kg)']       # 외부충격 (총반입량 베타 계수)
        )

        # MLE로 파라미터(분산, 베타, 파이 등) 학습
        res = model.fit(disp=False)

        # 칼만 스무딩으로 144개 전체 시점 예측
        smoothed_prices = res.predict()

        # 원래 가격이 비어있던(NaN) 곳만 스무딩 결과로 채워 넣기
        df_merged['대치가격(원)'] = df_merged['평균가격(원)'].fillna(smoothed_prices)

        # 결과 딕셔너리에 저장
        imputed_results[name] = df_merged

        print(f"[{name}] 칼만 대치 완료! (원본 {valid_count}건 -> 144건 복원)")

    except Exception as e:
        print(f"[{name}] 에러 발생: {e}")

print("✅ 모든 품목의 상태 공간 모델 결측치 대치가 완료되었습니다!")

[건고추_화건] 칼만 대치 완료! (원본 144건 -> 144건 복원)
[사과_후지] 칼만 대치 완료! (원본 125건 -> 144건 복원)
[감자_감자 수미] 칼만 대치 완료! (원본 144건 -> 144건 복원)
[배_신고] 칼만 대치 완료! (원본 144건 -> 144건 복원)
[깐마늘(국산)_깐마늘(국산)] 칼만 대치 완료! (원본 144건 -> 144건 복원)
[무_무] 칼만 대치 완료! (원본 144건 -> 144건 복원)
[상추_청] 칼만 대치 완료! (원본 144건 -> 144건 복원)
[배추_배추] 칼만 대치 완료! (원본 144건 -> 144건 복원)
[양파_양파] 칼만 대치 완료! (원본 144건 -> 144건 복원)
[대파_대파(일반)] 칼만 대치 완료! (원본 144건 -> 144건 복원)
✅ 모든 품목의 상태 공간 모델 결측치 대치가 완료되었습니다!


In [5]:
# ----------------------------------------------------
# [Step 5] 대치된 결과를 하나의 데이터프레임으로 병합 및 CSV 추출
# ----------------------------------------------------

final_dfs = []

# 딕셔너리에 저장된 10개 품목의 데이터프레임을 순회하며 합치기
for name, df in imputed_results.items():
    # name은 '건고추_화건' 형태이므로 '_'를 기준으로 분리
    item_name, item_kind = name.split('_')

    # 데이터프레임에 품목/품종 정보 추가 (나중에 보기 편하게)
    df_clean = df.copy()
    df_clean.insert(0, '품목명', item_name)
    df_clean.insert(1, '품종명', item_kind)

    # 리스트에 담기
    final_dfs.append(df_clean)

# 리스트에 있는 10개의 데이터프레임을 세로로 길게 하나로 합치기
final_imputed_data = pd.concat(final_dfs, ignore_index=True)

# 우리가 필요한 핵심 컬럼만 순서대로 정렬
final_imputed_data = final_imputed_data[['시점', '품목명', '품종명', '총반입량(kg)', '평균가격(원)', '대치가격(원)']]

# CSV 파일로 저장 (한글 깨짐 방지를 위해 utf-8-sig 사용)
output_filename = 'train_imputed_kalman.csv'
final_imputed_data.to_csv(output_filename, index=False, encoding='utf-8-sig')

print(f"🎉 모든 결측치 대치가 완료된 데이터가 '{output_filename}' 파일로 저장되었습니다!")

# 데이터가 잘 들어갔는지 상위 5개 확인
print(final_imputed_data.head())

🎉 모든 결측치 대치가 완료된 데이터가 'train_imputed_kalman.csv' 파일로 저장되었습니다!
         시점  품목명 품종명  총반입량(kg)   평균가격(원)   대치가격(원)
0  201801상순  건고추  화건       0.0  590000.0  590000.0
1  201801중순  건고추  화건       0.0  590000.0  590000.0
2  201801하순  건고추  화건       0.0  590000.0  590000.0
3  201802상순  건고추  화건       0.0  590000.0  590000.0
4  201802중순  건고추  화건       0.0  590000.0  590000.0


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# 1. 데이터 로드
train = pd.read_csv('train.csv')
wholesale_train = pd.read_csv('TRAIN_전국도매_2018-2021.csv')

test = pd.read_csv('TEST_00.csv')
wholesale_test = pd.read_csv('TEST_전국도매_00.csv')

# 2. 타겟 10대 품목
target_items = [
    {'품목명': '건고추', '품종명': '화건', '거래단위': '30 kg', '등급': '상품'},
    {'품목명': '사과', '품종명': '후지', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '감자', '품종명': '감자 수미', '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '배', '품종명': '신고', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '깐마늘(국산)', '품종명': '깐마늘(국산)', '거래단위': '20 kg', '등급': '상품'},
    {'품목명': '무', '품종명': '무', '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '상추', '품종명': '청', '거래단위': '100 g', '등급': '상품'},
    {'품목명': '배추', '품종명': '배추', '거래단위': '10키로망대', '등급': '상'},
    {'품목명': '양파', '품종명': '양파', '거래단위': '1키로', '등급': '상'},
    {'품목명': '대파', '품종명': '대파(일반)', '거래단위': '1키로단', '등급': '상'}
]

# 💡 [핵심] 도매시장 데이터용 이름표 매핑 (연료 주입을 위한 파이프 연결)
wholesale_name_map = {
    '깐마늘(국산)': '마늘',
    '건고추': '고추'
}

validation_results = []

for item in target_items:
    name = f"{item['품목명']}_{item['품종명']}"

    # --- [Step 1] 가격 데이터 추출 (기존과 동일하게 품종/단위/등급 모두 엄격히 매칭) ---
    cond_tr_p = (train['품목명'] == item['품목명']) & (train['품종명'] == item['품종명']) & \
                (train['거래단위'] == item['거래단위']) & (train['등급'] == item['등급'])
    df_tr_p = train[cond_tr_p][['시점', '평균가격(원)']]

    cond_te_p = (test['품목명'] == item['품목명']) & (test['품종명'] == item['품종명']) & \
                (test['거래단위'] == item['거래단위']) & (test['등급'] == item['등급'])
    df_te_p = test[cond_te_p][['시점', '평균가격(원)']]

    # 테스트셋에 해당 품목이나 'T' 시점 데이터가 없으면 패스 (IndexError 방지)
    if df_te_p.empty or df_te_p[df_te_p['시점'] == 'T'].empty:
        print(f"[{name}] Test 'T' 시점 데이터 부재. 건너뜁니다.")
        continue

    actual_price = df_te_p[df_te_p['시점'] == 'T']['평균가격(원)'].values[0]
    df_te_p.loc[df_te_p['시점'] == 'T', '평균가격(원)'] = np.nan # T시점 블라인드 처리

    # --- [Step 2] 반입량 데이터 추출 (💡 품종 무시! 오직 '품목명' 전체 합산) ---
    search_name = wholesale_name_map.get(item['품목명'], item['품목명'])

    # 도매시장 Train 반입량 합산
    cond_tr_s = (wholesale_train['품목명'] == search_name)
    df_tr_s = wholesale_train[cond_tr_s].groupby('시점')['총반입량(kg)'].sum().reset_index()

    # 도매시장 Test 반입량 합산
    cond_te_s = (wholesale_test['품목명'] == search_name)
    df_te_s = wholesale_test[cond_te_s].groupby('시점')['총반입량(kg)'].sum().reset_index()

    # --- [Step 3] 데이터 위아래 병합 및 결측치 0 처리 ---
    df_p = pd.concat([df_tr_p, df_te_p], ignore_index=True)
    df_s = pd.concat([df_tr_s, df_te_s], ignore_index=True)

    df_merged = pd.merge(df_p, df_s, on='시점', how='left')
    df_merged['총반입량(kg)'] = df_merged['총반입량(kg)'].fillna(0)

    # --- [Step 4] 상태 공간 모델 학습 및 예측 ---
    try:
        model = sm.tsa.UnobservedComponents(
            endog=df_merged['평균가격(원)'],
            level='local linear trend',
            seasonal=36,
            autoregressive=1,
            exog=df_merged['총반입량(kg)'] # 이제 모든 품종이 합산된 '진짜 연료'가 들어갑니다!
        )
        res = model.fit(disp=False)

        # 스무딩 및 T 시점(마지막 행) 예측값 추출
        smoothed_prices = res.predict()
        predicted_price = smoothed_prices.iloc[-1]

        error_rate = abs(predicted_price - actual_price) / actual_price * 100 if actual_price != 0 else 0

        validation_results.append({
            '품목': name,
            '실제_가격(T)': round(actual_price, 2),
            '예측_가격(T)': round(predicted_price, 2),
            '오차율(%)': round(error_rate, 2)
        })
        print(f"[{name}] 예측 완료! (오차율: {error_rate:.2f}%)")

    except Exception as e:
        print(f"[{name}] 모델 에러: {e}")

# --- [Step 5] 성적표 출력 ---
if validation_results:
    result_df = pd.DataFrame(validation_results)
    print("\n========== 🚀 대통합 공급량 기반 SSM 예측 성적표 ==========")
    print(result_df)
    print(f"\n💡 전체 품목 평균 예측 오차율(MAPE): {result_df['오차율(%)'].mean():.2f}%")

[건고추_화건] 예측 완료! (오차율: 7.13%)
[사과_후지] Test 'T' 시점 데이터 부재. 건너뜁니다.
[감자_감자 수미] 예측 완료! (오차율: 9.84%)
[배_신고] 예측 완료! (오차율: 17.42%)
[깐마늘(국산)_깐마늘(국산)] 예측 완료! (오차율: 0.56%)
[무_무] 예측 완료! (오차율: 2.21%)
[상추_청] 예측 완료! (오차율: 28.85%)
[배추_배추] 예측 완료! (오차율: 54.19%)


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


[양파_양파] 예측 완료! (오차율: 10.37%)
[대파_대파(일반)] 예측 완료! (오차율: 12.44%)

========== 🚀 대통합 공급량 기반 SSM 예측 성적표 ==========
                품목   실제_가격(T)   예측_가격(T)  오차율(%)
0           건고추_화건  673700.00  721723.93    7.13
1         감자_감자 수미   37901.38   41631.36    9.84
2             배_신고   30762.00   36119.93   17.42
3  깐마늘(국산)_깐마늘(국산)  168000.00  168932.80    0.56
4              무_무   34129.25   33375.07    2.21
5             상추_청    1213.00    1562.90   28.85
6            배추_배추   17092.50   26355.68   54.19
7            양파_양파    1513.38    1356.45   10.37
8        대파_대파(일반)    2097.38    1836.53   12.44

💡 전체 품목 평균 예측 오차율(MAPE): 15.89%


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [6]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings

warnings.filterwarnings('ignore')

# 1. 공통 학습 데이터 로드 (한 번만 로드해서 속도 향상)
train = pd.read_csv('train.csv')
wholesale_train = pd.read_csv('TRAIN_전국도매_2018-2021.csv')

# 2. 타겟 10대 품목 및 매핑
target_items = [
    {'품목명': '건고추', '품종명': '화건', '거래단위': '30 kg', '등급': '상품'},
    {'품목명': '사과', '품종명': '후지', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '감자', '품종명': '감자 수미', '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '배', '품종명': '신고', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '깐마늘(국산)', '품종명': '깐마늘(국산)', '거래단위': '20 kg', '등급': '상품'},
    {'품목명': '무', '품종명': '무', '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '상추', '품종명': '청', '거래단위': '100 g', '등급': '상품'},
    {'품목명': '배추', '품종명': '배추', '거래단위': '10키로망대', '등급': '상'},
    {'품목명': '양파', '품종명': '양파', '거래단위': '1키로', '등급': '상'},
    {'품목명': '대파', '품종명': '대파(일반)', '거래단위': '1키로단', '등급': '상'}
]
wholesale_name_map = {'깐마늘(국산)': '마늘', '건고추': '고추'}

# 전체 테스트 ID 생성 (TEST_00 ~ TEST_24)
test_ids = [f'TEST_{i:02d}' for i in range(25)]
all_validation_results = []

print(f"🚀 총 {len(test_ids)}개 테스트 파일 대상 NMAE 종합 모의고사 시작...\n")

# 3. 테스트 파일 루프
for test_id in test_ids:
    test_num = test_id.split('_')[1]

    try:
        test = pd.read_csv(f'{test_id}.csv')
        wholesale_test = pd.read_csv(f'TEST_전국도매_{test_num}.csv')
    except FileNotFoundError:
        print(f"  [경고] {test_id} 관련 파일을 찾을 수 없어 건너뜁니다.")
        continue

    print(f"🔄 [{test_id}] 예측 진행 중...")

    # 4. 품목별 모델링 및 예측
    for item in target_items:
        name = f"{item['품목명']}_{item['품종명']}"

        # --- 가격 데이터 ---
        cond_tr_p = (train['품목명'] == item['품목명']) & (train['품종명'] == item['품종명']) & \
                    (train['거래단위'] == item['거래단위']) & (train['등급'] == item['등급'])
        df_tr_p = train[cond_tr_p][['시점', '평균가격(원)']]

        cond_te_p = (test['품목명'] == item['품목명']) & (test['품종명'] == item['품종명']) & \
                    (test['거래단위'] == item['거래단위']) & (test['등급'] == item['등급'])
        df_te_p = test[cond_te_p][['시점', '평균가격(원)']]

        if df_te_p.empty or df_te_p[df_te_p['시점'] == 'T'].empty:
            continue

        # T 시점 실제 가격 저장 및 블라인드 처리
        actual_price = df_te_p[df_te_p['시점'] == 'T']['평균가격(원)'].values[0]
        df_te_p.loc[df_te_p['시점'] == 'T', '평균가격(원)'] = np.nan

        # --- 반입량 데이터 ---
        search_name = wholesale_name_map.get(item['품목명'], item['품목명'])
        cond_tr_s = (wholesale_train['품목명'] == search_name)
        df_tr_s = wholesale_train[cond_tr_s].groupby('시점')['총반입량(kg)'].sum().reset_index()

        cond_te_s = (wholesale_test['품목명'] == search_name)
        df_te_s = wholesale_test[cond_te_s].groupby('시점')['총반입량(kg)'].sum().reset_index()

        # --- 데이터 병합 ---
        df_p = pd.concat([df_tr_p, df_te_p], ignore_index=True)
        df_s = pd.concat([df_tr_s, df_te_s], ignore_index=True)
        df_merged = pd.merge(df_p, df_s, on='시점', how='left')
        df_merged['총반입량(kg)'] = df_merged['총반입량(kg)'].fillna(0)

        # --- 베이스라인 모델 학습 ---
        try:
            model = sm.tsa.UnobservedComponents(
                endog=df_merged['평균가격(원)'],
                level='local linear trend',
                seasonal=36,
                autoregressive=1,
                exog=df_merged['총반입량(kg)']
            )
            res = model.fit(disp=False)

            # 예측 및 오차 계산
            smoothed_prices = res.predict()
            predicted_price = max(smoothed_prices.iloc[-1], 0)

            abs_error = abs(actual_price - predicted_price)
            nmae = abs_error / actual_price if actual_price != 0 else 0

            all_validation_results.append({
                'Test_ID': test_id,
                '품목': name,
                'NMAE': nmae
            })

        except Exception as e:
            # 에러 발생 시 기록하지 않고 패스 (정확한 NMAE 평균을 위해)
            pass

# =======================================================================
# 5. 성적표 집계 및 출력
# =======================================================================
print("\n" + "="*50)
print(" 🚨 [종합] 25개 테스트셋 기반 평균 NMAE 성적표 (워스트 순)")
print("="*50)

if all_validation_results:
    result_df = pd.DataFrame(all_validation_results)

    # 품목별 평균 NMAE 계산
    summary_df = result_df.groupby('품목')['NMAE'].mean().reset_index()
    # 성적이 나쁜 순서대로 내림차순 정렬
    summary_df = summary_df.sort_values(by='NMAE', ascending=False).reset_index(drop=True)

    # 보기 좋게 소수점 4자리까지 포맷팅
    summary_df['NMAE'] = summary_df['NMAE'].round(4)

    print(summary_df.to_string(index=False))

    print("-" * 50)
    overall_mean_nmae = summary_df['NMAE'].mean()
    print(f"💡 25개 테스트셋 통합 평균 NMAE: {overall_mean_nmae:.4f}")

    if overall_mean_nmae <= 0.15:
        print("🎯 완벽합니다! 바로 제출하셔도 좋은 수준입니다.")
    elif overall_mean_nmae <= 0.20:
        print("🔥 아주 좋습니다! 위에 있는 워스트 품목 1~2개만 집중 튜닝합시다.")
    else:
        print("🛠️ 모델 스케일링이나 외생변수 처리에 대대적인 보완이 필요합니다.")
else:
    print("수집된 결과가 없습니다. 에러 로그를 확인해주세요.")

🚀 총 25개 테스트 파일 대상 NMAE 종합 모의고사 시작...

🔄 [TEST_00] 예측 진행 중...
🔄 [TEST_01] 예측 진행 중...
🔄 [TEST_02] 예측 진행 중...
🔄 [TEST_03] 예측 진행 중...
🔄 [TEST_04] 예측 진행 중...
🔄 [TEST_05] 예측 진행 중...
🔄 [TEST_06] 예측 진행 중...
🔄 [TEST_07] 예측 진행 중...
🔄 [TEST_08] 예측 진행 중...
🔄 [TEST_09] 예측 진행 중...
🔄 [TEST_10] 예측 진행 중...
🔄 [TEST_11] 예측 진행 중...
🔄 [TEST_12] 예측 진행 중...
🔄 [TEST_13] 예측 진행 중...
🔄 [TEST_14] 예측 진행 중...
🔄 [TEST_15] 예측 진행 중...
🔄 [TEST_16] 예측 진행 중...
🔄 [TEST_17] 예측 진행 중...
🔄 [TEST_18] 예측 진행 중...
🔄 [TEST_19] 예측 진행 중...
🔄 [TEST_20] 예측 진행 중...
🔄 [TEST_21] 예측 진행 중...
🔄 [TEST_22] 예측 진행 중...
🔄 [TEST_23] 예측 진행 중...
🔄 [TEST_24] 예측 진행 중...

 🚨 [종합] 25개 테스트셋 기반 평균 NMAE 성적표 (워스트 순)
             품목   NMAE
          배추_배추 0.2674
      대파_대파(일반) 0.1894
           상추_청 0.1572
            무_무 0.1506
          양파_양파 0.1500
       감자_감자 수미 0.1309
           배_신고 0.0501
          사과_후지 0.0437
         건고추_화건 0.0247
깐마늘(국산)_깐마늘(국산) 0.0072
--------------------------------------------------
💡 25개 테스트셋 통합 평균 NMAE: 0.1171
🎯 완벽합니다! 바로 

건고추 >> 마늘, 3시점 반입량 영향 후 오류

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings

warnings.filterwarnings('ignore')

# 1. 공통 학습 데이터 로드
train = pd.read_csv('train.csv')
wholesale_train = pd.read_csv('TRAIN_전국도매_2018-2021.csv')

# 2. 타겟 10대 품목
target_items = [
    {'품목명': '건고추', '품종명': '화건', '거래단위': '30 kg', '등급': '상품'},
    {'품목명': '사과', '품종명': '후지', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '감자', '품종명': '감자 수미', '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '배', '품종명': '신고', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '깐마늘(국산)', '품종명': '깐마늘(국산)', '거래단위': '20 kg', '등급': '상품'},
    {'품목명': '무', '품종명': '무', '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '상추', '품종명': '청', '거래단위': '100 g', '등급': '상품'},
    {'품목명': '배추', '품종명': '배추', '거래단위': '10키로망대', '등급': '상'},
    {'품목명': '양파', '품종명': '양파', '거래단위': '1키로', '등급': '상'},
    {'품목명': '대파', '품종명': '대파(일반)', '거래단위': '1키로단', '등급': '상'}
]

# 💡 [핵심 1] 건고추의 반입량을 '마늘'로 대치하도록 매핑 수정!
wholesale_name_map = {
    '깐마늘(국산)': '마늘',
    '건고추': '마늘'  # 고추 -> 마늘로 완벽하게 덮어씌움
}

# 전체 테스트 ID 생성 (TEST_00 ~ TEST_24)
test_ids = [f'TEST_{i:02d}' for i in range(25)]
all_validation_results = []

print(f"🚀 총 {len(test_ids)}개 테스트 파일 대상 NMAE 종합 모의고사 시작...\n")

# 3. 테스트 파일 루프
for test_id in test_ids:
    test_num = test_id.split('_')[1]

    try:
        test = pd.read_csv(f'{test_id}.csv')
        wholesale_test = pd.read_csv(f'TEST_전국도매_{test_num}.csv')
    except FileNotFoundError:
        continue

    print(f"🔄 [{test_id}] 예측 진행 중...")

    # 4. 품목별 모델링 및 예측
    for item in target_items:
        name = f"{item['품목명']}_{item['품종명']}"

        # --- 가격 데이터 ---
        cond_tr_p = (train['품목명'] == item['품목명']) & (train['품종명'] == item['품종명']) & \
                    (train['거래단위'] == item['거래단위']) & (train['등급'] == item['등급'])
        df_tr_p = train[cond_tr_p][['시점', '평균가격(원)']]

        cond_te_p = (test['품목명'] == item['품목명']) & (test['품종명'] == item['품종명']) & \
                    (test['거래단위'] == item['거래단위']) & (test['등급'] == item['등급'])
        df_te_p = test[cond_te_p][['시점', '평균가격(원)']]

        if df_te_p.empty or df_te_p[df_te_p['시점'] == 'T'].empty:
            continue

        # T 시점 실제 가격 저장 및 블라인드 처리
        actual_price = df_te_p[df_te_p['시점'] == 'T']['평균가격(원)'].values[0]
        df_te_p.loc[df_te_p['시점'] == 'T', '평균가격(원)'] = np.nan

        # --- 반입량 데이터 ---
        search_name = wholesale_name_map.get(item['품목명'], item['품목명'])
        cond_tr_s = (wholesale_train['품목명'] == search_name)
        df_tr_s = wholesale_train[cond_tr_s].groupby('시점')['총반입량(kg)'].sum().reset_index()

        cond_te_s = (wholesale_test['품목명'] == search_name)
        df_te_s = wholesale_test[cond_te_s].groupby('시점')['총반입량(kg)'].sum().reset_index()

        # --- 데이터 병합 ---
        df_p = pd.concat([df_tr_p, df_te_p], ignore_index=True)
        df_s = pd.concat([df_tr_s, df_te_s], ignore_index=True)
        df_merged = pd.merge(df_p, df_s, on='시점', how='left')

        # 반입량 결측치 0 처리
        df_merged['총반입량(kg)'] = df_merged['총반입량(kg)'].fillna(0)

        # 💡 [핵심 2] 시차(Lag) 3시점 적용!
        # 현재 시점 가격을 맞추기 위해 3시점 전의 물량을 끌어내림
        df_merged['exog_lag3'] = df_merged['총반입량(kg)'].shift(2)
        # shift로 인해 발생하는 초반 3개의 결측치는 그 이후 첫 번째 값으로 뒤로 채움(bfill)
        df_merged['exog_lag3'] = df_merged['exog_lag3'].fillna(method='bfill')

        # --- 베이스라인 모델 학습 ---
        try:
            model = sm.tsa.UnobservedComponents(
                endog=df_merged['평균가격(원)'],
                level='local linear trend',
                seasonal=36,
                autoregressive=1,
                exog=df_merged['exog_lag3'] # 🚀 3시점 밀린 물량 투입!
            )
            res = model.fit(disp=False)

            # 예측 및 오차 계산
            smoothed_prices = res.predict()
            predicted_price = max(smoothed_prices.iloc[-1], 0)

            abs_error = abs(actual_price - predicted_price)
            nmae = abs_error / actual_price if actual_price != 0 else 0

            all_validation_results.append({
                'Test_ID': test_id,
                '품목': name,
                'NMAE': nmae
            })

        except Exception as e:
            pass

# =======================================================================
# 5. 성적표 집계 및 출력
# =======================================================================
print("\n" + "="*50)
print(" 🚨 [종합] 25개 테스트셋 기반 평균 NMAE 성적표 (시차 3 적용)")
print("="*50)

if all_validation_results:
    result_df = pd.DataFrame(all_validation_results)

    # 품목별 평균 NMAE 계산
    summary_df = result_df.groupby('품목')['NMAE'].mean().reset_index()
    summary_df = summary_df.sort_values(by='NMAE', ascending=False).reset_index(drop=True)
    summary_df['NMAE'] = summary_df['NMAE'].round(4)

    print(summary_df.to_string(index=False))

    print("-" * 50)
    overall_mean_nmae = summary_df['NMAE'].mean()
    print(f"💡 25개 테스트셋 통합 평균 NMAE: {overall_mean_nmae:.4f}")
else:
    print("수집된 결과가 없습니다. 에러 로그를 확인해주세요.")

🚀 총 25개 테스트 파일 대상 NMAE 종합 모의고사 시작...

🔄 [TEST_00] 예측 진행 중...
🔄 [TEST_01] 예측 진행 중...
🔄 [TEST_02] 예측 진행 중...
🔄 [TEST_03] 예측 진행 중...
🔄 [TEST_04] 예측 진행 중...
🔄 [TEST_05] 예측 진행 중...
🔄 [TEST_06] 예측 진행 중...
🔄 [TEST_07] 예측 진행 중...
🔄 [TEST_08] 예측 진행 중...
🔄 [TEST_09] 예측 진행 중...
🔄 [TEST_10] 예측 진행 중...
🔄 [TEST_11] 예측 진행 중...
🔄 [TEST_12] 예측 진행 중...
🔄 [TEST_13] 예측 진행 중...
🔄 [TEST_14] 예측 진행 중...
🔄 [TEST_15] 예측 진행 중...
🔄 [TEST_16] 예측 진행 중...
🔄 [TEST_17] 예측 진행 중...
🔄 [TEST_18] 예측 진행 중...
🔄 [TEST_19] 예측 진행 중...
🔄 [TEST_20] 예측 진행 중...
🔄 [TEST_21] 예측 진행 중...
🔄 [TEST_22] 예측 진행 중...
🔄 [TEST_23] 예측 진행 중...
🔄 [TEST_24] 예측 진행 중...

 🚨 [종합] 25개 테스트셋 기반 평균 NMAE 성적표 (시차 3 적용)
             품목   NMAE
          배추_배추 0.2689
      대파_대파(일반) 0.1772
          양파_양파 0.1549
           상추_청 0.1500
            무_무 0.1306
       감자_감자 수미 0.1256
           배_신고 0.0538
          사과_후지 0.0414
         건고추_화건 0.0252
깐마늘(국산)_깐마늘(국산) 0.0073
--------------------------------------------------
💡 25개 테스트셋 통합 평균 NMAE: 0.1135


배추만 다시보자

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings

warnings.filterwarnings('ignore')

# 1. 공통 학습 데이터 로드
train = pd.read_csv('train.csv')
wholesale_train = pd.read_csv('TRAIN_전국도매_2018-2021.csv')

# 2. 타겟 10대 품목 및 매핑
target_items = [
    {'품목명': '건고추', '품종명': '화건', '거래단위': '30 kg', '등급': '상품'},
    {'품목명': '사과', '품종명': '후지', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '감자', '품종명': '감자 수미', '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '배', '품종명': '신고', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '깐마늘(국산)', '품종명': '깐마늘(국산)', '거래단위': '20 kg', '등급': '상품'},
    {'품목명': '무', '품종명': '무', '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '상추', '품종명': '청', '거래단위': '100 g', '등급': '상품'},
    {'품목명': '배추', '품종명': '배추', '거래단위': '10키로망대', '등급': '상'},
    {'품목명': '양파', '품종명': '양파', '거래단위': '1키로', '등급': '상'},
    {'품목명': '대파', '품종명': '대파(일반)', '거래단위': '1키로단', '등급': '상'}
]
wholesale_name_map = {'깐마늘(국산)': '마늘', '건고추': '마늘'} # 건고추->마늘 대치 유지

test_ids = [f'TEST_{i:02d}' for i in range(25)]
all_validation_results = []

print(f"🚀 총 {len(test_ids)}개 테스트 파일 대상 모의고사 시작 (배추 특화 튜닝 적용)...\n")

for test_id in test_ids:
    test_num = test_id.split('_')[1]

    try:
        test = pd.read_csv(f'{test_id}.csv')
        wholesale_test = pd.read_csv(f'TEST_전국도매_{test_num}.csv')
    except FileNotFoundError:
        continue

    print(f"🔄 [{test_id}] 예측 진행 중...")

    for item in target_items:
        name = f"{item['품목명']}_{item['품종명']}"

        # --- 가격 데이터 ---
        cond_tr_p = (train['품목명'] == item['품목명']) & (train['품종명'] == item['품종명']) & \
                    (train['거래단위'] == item['거래단위']) & (train['등급'] == item['등급'])
        df_tr_p = train[cond_tr_p][['시점', '평균가격(원)']]

        cond_te_p = (test['품목명'] == item['품목명']) & (test['품종명'] == item['품종명']) & \
                    (test['거래단위'] == item['거래단위']) & (test['등급'] == item['등급'])
        df_te_p = test[cond_te_p][['시점', '평균가격(원)']]

        if df_te_p.empty or df_te_p[df_te_p['시점'] == 'T'].empty:
            continue

        actual_price = df_te_p[df_te_p['시점'] == 'T']['평균가격(원)'].values[0]
        df_te_p.loc[df_te_p['시점'] == 'T', '평균가격(원)'] = np.nan

        # --- 반입량 데이터 ---
        search_name = wholesale_name_map.get(item['품목명'], item['품목명'])
        cond_tr_s = (wholesale_train['품목명'] == search_name)
        df_tr_s = wholesale_train[cond_tr_s].groupby('시점')['총반입량(kg)'].sum().reset_index()

        cond_te_s = (wholesale_test['품목명'] == search_name)
        df_te_s = wholesale_test[cond_te_s].groupby('시점')['총반입량(kg)'].sum().reset_index()

        # --- 데이터 병합 및 시차(Lag 2) 적용 ---
        df_p = pd.concat([df_tr_p, df_te_p], ignore_index=True)
        df_s = pd.concat([df_tr_s, df_te_s], ignore_index=True)
        df_merged = pd.merge(df_p, df_s, on='시점', how='left')

        df_merged['총반입량(kg)'] = df_merged['총반입량(kg)'].fillna(0)

        # 💡 사용자님이 찾은 최적의 시차 2시점 적용
        df_merged['exog_lag2'] = df_merged['총반입량(kg)'].shift(2).fillna(method='bfill')

        # --- 모델 학습 ---
        try:
            # 💡 [핵심] 배추일 때만 튜닝된 파라미터 적용
            if name == '배추_배추':
                model = sm.tsa.UnobservedComponents(
                    endog=df_merged['평균가격(원)'],
                    level='local linear trend',
                    stochastic_level=True,       # 기준 레벨 변동 허용 (급등락 대응)
                    stochastic_slope=False,      # 장기 기울기 고정 (불필요한 왜곡 방지)
                    freq_seasonal=[{'period': 36, 'harmonics': 3}], # 뾰족한 여름 피크용 파동
                    autoregressive=2,            # AR(2)로 급등세 관성 기억력 2배 확장
                    exog=df_merged['exog_lag2']
                )
            # 그 외 9개 품목은 기존 베이스라인(Lag 2) 유지
            else:
                model = sm.tsa.UnobservedComponents(
                    endog=df_merged['평균가격(원)'],
                    level='local linear trend',
                    seasonal=36,
                    autoregressive=1,
                    exog=df_merged['exog_lag2']
                )

            res = model.fit(disp=False)

            # 예측 및 오차 계산
            smoothed_prices = res.predict()
            predicted_price = max(smoothed_prices.iloc[-1], 0)

            abs_error = abs(actual_price - predicted_price)
            nmae = abs_error / actual_price if actual_price != 0 else 0

            all_validation_results.append({
                'Test_ID': test_id,
                '품목': name,
                'NMAE': nmae
            })

        except Exception as e:
            pass

# =======================================================================
# 5. 성적표 집계 및 출력
# =======================================================================
print("\n" + "="*50)
print(" 🚨 [종합] 배추 특화 튜닝 반영 NMAE 성적표")
print("="*50)

if all_validation_results:
    result_df = pd.DataFrame(all_validation_results)
    summary_df = result_df.groupby('품목')['NMAE'].mean().reset_index()
    summary_df = summary_df.sort_values(by='NMAE', ascending=False).reset_index(drop=True)
    summary_df['NMAE'] = summary_df['NMAE'].round(4)

    print(summary_df.to_string(index=False))

    print("-" * 50)
    overall_mean_nmae = summary_df['NMAE'].mean()
    print(f"💡 25개 테스트셋 통합 평균 NMAE: {overall_mean_nmae:.4f}")
else:
    print("수집된 결과가 없습니다. 에러 로그를 확인해주세요.")

🚀 총 25개 테스트 파일 대상 모의고사 시작 (배추 특화 튜닝 적용)...

🔄 [TEST_00] 예측 진행 중...
🔄 [TEST_01] 예측 진행 중...
🔄 [TEST_02] 예측 진행 중...
🔄 [TEST_03] 예측 진행 중...
🔄 [TEST_04] 예측 진행 중...
🔄 [TEST_05] 예측 진행 중...
🔄 [TEST_06] 예측 진행 중...
🔄 [TEST_07] 예측 진행 중...
🔄 [TEST_08] 예측 진행 중...
🔄 [TEST_09] 예측 진행 중...
🔄 [TEST_10] 예측 진행 중...
🔄 [TEST_11] 예측 진행 중...
🔄 [TEST_12] 예측 진행 중...
🔄 [TEST_13] 예측 진행 중...
🔄 [TEST_14] 예측 진행 중...
🔄 [TEST_15] 예측 진행 중...
🔄 [TEST_16] 예측 진행 중...
🔄 [TEST_17] 예측 진행 중...
🔄 [TEST_18] 예측 진행 중...
🔄 [TEST_19] 예측 진행 중...
🔄 [TEST_20] 예측 진행 중...
🔄 [TEST_21] 예측 진행 중...
🔄 [TEST_22] 예측 진행 중...
🔄 [TEST_23] 예측 진행 중...
🔄 [TEST_24] 예측 진행 중...

 🚨 [종합] 배추 특화 튜닝 반영 NMAE 성적표
             품목   NMAE
          배추_배추 0.1918
      대파_대파(일반) 0.1772
          양파_양파 0.1549
           상추_청 0.1500
            무_무 0.1306
       감자_감자 수미 0.1256
           배_신고 0.0538
          사과_후지 0.0414
         건고추_화건 0.0252
깐마늘(국산)_깐마늘(국산) 0.0073
--------------------------------------------------
💡 25개 테스트셋 통합 평균 NMAE: 0.1058


상추랑 대파까지 건드리기

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings

warnings.filterwarnings('ignore')

# =======================================================================
# 1. 공통 학습 데이터 로드
# =======================================================================
train = pd.read_csv('train.csv')
wholesale_train = pd.read_csv('TRAIN_전국도매_2018-2021.csv')

# 2. 타겟 10대 품목 및 매핑
target_items = [
    {'품목명': '건고추', '품종명': '화건', '거래단위': '30 kg', '등급': '상품'},
    {'품목명': '사과', '품종명': '후지', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '감자', '품종명': '감자 수미', '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '배', '품종명': '신고', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '깐마늘(국산)', '품종명': '깐마늘(국산)', '거래단위': '20 kg', '등급': '상품'},
    {'품목명': '무', '품종명': '무', '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '상추', '품종명': '청', '거래단위': '100 g', '등급': '상품'},
    {'품목명': '배추', '품종명': '배추', '거래단위': '10키로망대', '등급': '상'},
    {'품목명': '양파', '품종명': '양파', '거래단위': '1키로', '등급': '상'},
    {'품목명': '대파', '품종명': '대파(일반)', '거래단위': '1키로단', '등급': '상'}
]

# 💡 [핵심] 건고추 반입량을 '마늘'로 대치
wholesale_name_map = {'깐마늘(국산)': '마늘', '건고추': '마늘'}

test_ids = [f'TEST_{i:02d}' for i in range(25)]
all_validation_results = []

print(f"🚀 총 {len(test_ids)}개 테스트 파일 대상 모의고사 시작 (배추/상추/대파 맞춤형 튜닝)...\n")

# =======================================================================
# 3. 테스트 파일 루프
# =======================================================================
for test_id in test_ids:
    test_num = test_id.split('_')[1]

    try:
        test = pd.read_csv(f'{test_id}.csv')
        wholesale_test = pd.read_csv(f'TEST_전국도매_{test_num}.csv')
    except FileNotFoundError:
        continue

    print(f"🔄 [{test_id}] 예측 진행 중...")

    # 4. 품목별 모델링 및 예측
    for item in target_items:
        name = f"{item['품목명']}_{item['품종명']}"

        # --- 가격 데이터 ---
        cond_tr_p = (train['품목명'] == item['품목명']) & (train['품종명'] == item['품종명']) & \
                    (train['거래단위'] == item['거래단위']) & (train['등급'] == item['등급'])
        df_tr_p = train[cond_tr_p][['시점', '평균가격(원)']]

        cond_te_p = (test['품목명'] == item['품목명']) & (test['품종명'] == item['품종명']) & \
                    (test['거래단위'] == item['거래단위']) & (test['등급'] == item['등급'])
        df_te_p = test[cond_te_p][['시점', '평균가격(원)']]

        if df_te_p.empty or df_te_p[df_te_p['시점'] == 'T'].empty:
            continue

        # T 시점 실제 가격 저장 및 블라인드 처리
        actual_price = df_te_p[df_te_p['시점'] == 'T']['평균가격(원)'].values[0]
        df_te_p.loc[df_te_p['시점'] == 'T', '평균가격(원)'] = np.nan

        # --- 반입량 데이터 ---
        search_name = wholesale_name_map.get(item['품목명'], item['품목명'])
        cond_tr_s = (wholesale_train['품목명'] == search_name)
        df_tr_s = wholesale_train[cond_tr_s].groupby('시점')['총반입량(kg)'].sum().reset_index()

        cond_te_s = (wholesale_test['품목명'] == search_name)
        df_te_s = wholesale_test[cond_te_s].groupby('시점')['총반입량(kg)'].sum().reset_index()

        # --- 데이터 병합 ---
        df_p = pd.concat([df_tr_p, df_te_p], ignore_index=True)
        df_s = pd.concat([df_tr_s, df_te_s], ignore_index=True)
        df_merged = pd.merge(df_p, df_s, on='시점', how='left')

        # 반입량 결측치 0 처리
        df_merged['총반입량(kg)'] = df_merged['총반입량(kg)'].fillna(0)

        # 💡 시차 2시점 (Lag 2) 적용 및 단위 스케일링(/10000)
        df_merged['exog_lag2'] = df_merged['총반입량(kg)'].shift(2).fillna(method='bfill') / 10000

        # --- 베이스라인 모델 학습 ---
        try:
            # 💡 [맞춤형 엔진 1] 배추 (피크/관성형)
            if name == '배추_배추':
                model = sm.tsa.UnobservedComponents(
                    endog=df_merged['평균가격(원)'],
                    level='local linear trend',
                    stochastic_level=True,
                    stochastic_slope=False,
                    freq_seasonal=[{'period': 36, 'harmonics': 3}],
                    autoregressive=2,
                    exog=df_merged['exog_lag2']
                )

            # 💡 [맞춤형 엔진 2] 상추 (강력한 주기성/추세 배제)
            elif name == '상추_청':
                model = sm.tsa.UnobservedComponents(
                    endog=df_merged['평균가격(원)'],
                    level='local linear trend',
                    stochastic_level=True,
                    stochastic_slope=False,
                    seasonal=36,
                    autoregressive=1,
                    exog=df_merged['exog_lag2']
                )

            # 💡 [맞춤형 엔진 3] 대파 (랜덤 충격/수준 점프형)
            elif name == '대파_대파(일반)':
                model = sm.tsa.UnobservedComponents(
                    endog=df_merged['평균가격(원)'],
                    level='local linear trend',
                    stochastic_level=True,
                    stochastic_slope=False,
                    seasonal=36,
                    autoregressive=1,
                    exog=df_merged['exog_lag2']
                )

            # 💡 [기본 엔진] 나머지 품목
            else:
                model = sm.tsa.UnobservedComponents(
                    endog=df_merged['평균가격(원)'],
                    level='local linear trend',
                    seasonal=36,
                    autoregressive=1,
                    exog=df_merged['exog_lag2']
                )

            res = model.fit(disp=False)

            # 예측 및 오차 계산
            smoothed_prices = res.predict()
            predicted_price = max(smoothed_prices.iloc[-1], 0)

            abs_error = abs(actual_price - predicted_price)
            nmae = abs_error / actual_price if actual_price != 0 else 0

            all_validation_results.append({
                'Test_ID': test_id,
                '품목': name,
                'NMAE': nmae
            })

        except Exception as e:
            pass

# =======================================================================
# 5. 성적표 집계 및 출력
# =======================================================================
print("\n" + "="*50)
print(" 🚨 [종합] 배추/상추/대파 맞춤 튜닝 반영 NMAE 성적표")
print("="*50)

if all_validation_results:
    result_df = pd.DataFrame(all_validation_results)

    summary_df = result_df.groupby('품목')['NMAE'].mean().reset_index()
    summary_df = summary_df.sort_values(by='NMAE', ascending=False).reset_index(drop=True)
    summary_df['NMAE'] = summary_df['NMAE'].round(4)

    print(summary_df.to_string(index=False))

    print("-" * 50)
    overall_mean_nmae = summary_df['NMAE'].mean()
    print(f"💡 25개 테스트셋 통합 평균 NMAE: {overall_mean_nmae:.4f}")

    if overall_mean_nmae <= 0.15:
        print("🎯 완벽합니다! 목표 궤도 진입!")
else:
    print("수집된 결과가 없습니다. 에러 로그를 확인해주세요.")

🚀 총 25개 테스트 파일 대상 모의고사 시작 (배추/상추/대파 맞춤형 튜닝)...

🔄 [TEST_00] 예측 진행 중...
🔄 [TEST_01] 예측 진행 중...
🔄 [TEST_02] 예측 진행 중...
🔄 [TEST_03] 예측 진행 중...
🔄 [TEST_04] 예측 진행 중...
🔄 [TEST_05] 예측 진행 중...
🔄 [TEST_06] 예측 진행 중...
🔄 [TEST_07] 예측 진행 중...
🔄 [TEST_08] 예측 진행 중...
🔄 [TEST_09] 예측 진행 중...
🔄 [TEST_10] 예측 진행 중...
🔄 [TEST_11] 예측 진행 중...
🔄 [TEST_12] 예측 진행 중...
🔄 [TEST_13] 예측 진행 중...
🔄 [TEST_14] 예측 진행 중...
🔄 [TEST_15] 예측 진행 중...
🔄 [TEST_16] 예측 진행 중...
🔄 [TEST_17] 예측 진행 중...
🔄 [TEST_18] 예측 진행 중...
🔄 [TEST_19] 예측 진행 중...
🔄 [TEST_20] 예측 진행 중...
🔄 [TEST_21] 예측 진행 중...
🔄 [TEST_22] 예측 진행 중...
🔄 [TEST_23] 예측 진행 중...
🔄 [TEST_24] 예측 진행 중...

 🚨 [종합] 배추/상추/대파 맞춤 튜닝 반영 NMAE 성적표
             품목   NMAE
          배추_배추 0.1893
          양파_양파 0.1576
            무_무 0.1341
      대파_대파(일반) 0.1235
           상추_청 0.1026
       감자_감자 수미 0.0943
           배_신고 0.0528
          사과_후지 0.0467
         건고추_화건 0.0248
깐마늘(국산)_깐마늘(국산) 0.0073
--------------------------------------------------
💡 25개 테스트셋 통합 평균 NMAE: 0.0933
🎯 완벽합니다

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings

# 모델 학습 시 발생하는 경고 메시지를 숨겨서 출력을 깔끔하게 만듭니다.
warnings.filterwarnings('ignore')

# 1. 공통 데이터 및 제출 양식 로드
train = pd.read_csv('train.csv')
wholesale_train = pd.read_csv('TRAIN_전국도매_2018-2021.csv')
submission = pd.read_csv('sample_submission.csv')

# 2. 타겟 10대 품목 및 도매시장 매핑 (작성하신 내용과 동일)
target_items = [
    {'품목명': '건고추', '품종명': '화건', '거래단위': '30 kg', '등급': '상품'},
    {'품목명': '사과', '품종명': '후지', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '감자', '품종명': '감자 수미', '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '배', '품종명': '신고', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '깐마늘(국산)', '품종명': '깐마늘(국산)', '거래단위': '20 kg', '등급': '상품'},
    {'품목명': '무', '품종명': '무', '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '상추', '품종명': '청', '거래단위': '100 g', '등급': '상품'},
    {'품목명': '배추', '품종명': '배추', '거래단위': '10키로망대', '등급': '상'},
    {'품목명': '양파', '품종명': '양파', '거래단위': '1키로', '등급': '상'},
    {'품목명': '대파', '품종명': '대파(일반)', '거래단위': '1키로단', '등급': '상'}
]

wholesale_name_map = {
    '깐마늘(국산)': '마늘',
    '건고추': '고추'
}

# 3. 제출 양식에서 평가해야 할 TEST ID 목록 추출 (예: 'TEST_00', 'TEST_01', ...)
test_ids = submission['시점'].apply(lambda x: x.split('+')[0]).unique()

# 결과를 채워넣기 편하도록 '시점'을 인덱스로 설정합니다.
result_sub = submission.copy()
result_sub.set_index('시점', inplace=True)

print(f"총 {len(test_ids)}개의 테스트 데이터셋에 대한 예측을 시작합니다...\n")

# 4. 각 테스트 데이터셋별로 반복 처리
for test_id in test_ids:
    print(f"[{test_id}] 예측 진행 중...")

    # 해당 번호의 테스트 데이터 및 도매 데이터 로드
    # 예: test_id가 'TEST_00'이면 'TEST_00.csv' 와 'TEST_전국도매_00.csv'를 로드
    test_num = test_id.split('_')[1] # '00', '01' 등 추출

    try:
        test = pd.read_csv(f'{test_id}.csv')
        wholesale_test = pd.read_csv(f'TEST_전국도매_{test_num}.csv')
    except FileNotFoundError:
        print(f"  [경고] {test_id} 관련 파일을 찾을 수 없어 건너뜁니다.")
        continue

    # 5. 각 품목별로 모델 학습 및 3-step 예측
    for item in target_items:
        # 제출 파일의 컬럼명은 품목명과 동일합니다. (예: '사과', '배추')
        col_name = item['품목명']

        # --- [Step 5-1] 가격 데이터 추출 (Train + Test) ---
        cond_tr_p = (train['품목명'] == item['품목명']) & (train['품종명'] == item['품종명']) & \
                    (train['거래단위'] == item['거래단위']) & (train['등급'] == item['등급'])
        df_tr_p = train[cond_tr_p][['시점', '평균가격(원)']]

        cond_te_p = (test['품목명'] == item['품목명']) & (test['품종명'] == item['품종명']) & \
                    (test['거래단위'] == item['거래단위']) & (test['등급'] == item['등급'])
        df_te_p = test[cond_te_p][['시점', '평균가격(원)']]

        df_p = pd.concat([df_tr_p, df_te_p], ignore_index=True)

        # --- [Step 5-2] 반입량 데이터 추출 및 합산 ---
        search_name = wholesale_name_map.get(item['품목명'], item['품목명'])

        cond_tr_s = (wholesale_train['품목명'] == search_name)
        df_tr_s = wholesale_train[cond_tr_s].groupby('시점')['총반입량(kg)'].sum().reset_index()

        cond_te_s = (wholesale_test['품목명'] == search_name)
        df_te_s = wholesale_test[cond_te_s].groupby('시점')['총반입량(kg)'].sum().reset_index()

        df_s = pd.concat([df_tr_s, df_te_s], ignore_index=True)

        # --- [Step 5-3] 병합 및 결측치 처리 ---
        df_merged = pd.merge(df_p, df_s, on='시점', how='left')
        df_merged['총반입량(kg)'] = df_merged['총반입량(kg)'].fillna(0)
        # 가격 결측치는 SSM이 알아서 처리하므로 따로 fillna 하지 않습니다.

        # --- [Step 5-4] 미래 예측을 위한 외부 변수(Exog) 가정 ---
        # T+1, T+2, T+3 시점의 가격을 예측하려면 해당 시점의 '총반입량' 정보도 필요합니다.
        # 가장 단순하면서 합리적인 방법으로, "최근 3개 시점의 평균 반입량"이 미래 3번의 시점에도 유지된다고 가정합니다.
        recent_mean_supply = df_merged['총반입량(kg)'].tail(3).mean()
        future_exog = [recent_mean_supply, recent_mean_supply, recent_mean_supply]

        # --- [Step 5-5] 모델 학습 및 예측 ---
        try:
            model = sm.tsa.UnobservedComponents(
                endog=df_merged['평균가격(원)'],
                level='local linear trend',
                seasonal=36,
                autoregressive=1,
                exog=df_merged['총반입량(kg)']
            )
            res = model.fit(disp=False)

            # forecast() 함수를 사용하여 데이터 끝점에서 3스텝(T+1, T+2, T+3) 미래 예측
            forecasts = res.forecast(steps=3, exog=future_exog)

            # 음수 가격 방지 (가격이 0 미만이 될 수 없으므로)
            forecasts = np.maximum(forecasts, 0)

            # --- [Step 5-6] 예측값을 제출 양식 데이터프레임에 매핑 ---
            result_sub.loc[f'{test_id}+1순', col_name] = forecasts.iloc[0]
            result_sub.loc[f'{test_id}+2순', col_name] = forecasts.iloc[1]
            result_sub.loc[f'{test_id}+3순', col_name] = forecasts.iloc[2]

        except Exception as e:
            print(f"  [오류] {test_id} - {col_name} 예측 실패: {e}")
            # 에러 발생 시 베이스라인으로 0으로 채움 (대회 규정 및 상황에 맞게 변경 가능)
            result_sub.loc[f'{test_id}+1순', col_name] = 0
            result_sub.loc[f'{test_id}+2순', col_name] = 0
            result_sub.loc[f'{test_id}+3순', col_name] = 0

# 6. 인덱스를 다시 컬럼으로 빼고 최종 CSV 저장
result_sub.reset_index(inplace=True)
result_sub.to_csv('final_submission.csv', index=False)
print("\n🎉 모든 예측이 완료되었으며 'final_submission.csv' 파일이 성공적으로 저장되었습니다!")

총 25개의 테스트 데이터셋에 대한 예측을 시작합니다...

[TEST_00] 예측 진행 중...
[TEST_01] 예측 진행 중...
[TEST_02] 예측 진행 중...
[TEST_03] 예측 진행 중...
[TEST_04] 예측 진행 중...
[TEST_05] 예측 진행 중...
[TEST_06] 예측 진행 중...
[TEST_07] 예측 진행 중...
[TEST_08] 예측 진행 중...
[TEST_09] 예측 진행 중...
[TEST_10] 예측 진행 중...
[TEST_11] 예측 진행 중...
[TEST_12] 예측 진행 중...
[TEST_13] 예측 진행 중...
[TEST_14] 예측 진행 중...
[TEST_15] 예측 진행 중...
[TEST_16] 예측 진행 중...
[TEST_17] 예측 진행 중...
[TEST_18] 예측 진행 중...
[TEST_19] 예측 진행 중...
[TEST_20] 예측 진행 중...
[TEST_21] 예측 진행 중...
[TEST_22] 예측 진행 중...
[TEST_23] 예측 진행 중...
[TEST_24] 예측 진행 중...

🎉 모든 예측이 완료되었으며 'final_submission.csv' 파일이 성공적으로 저장되었습니다!


무 배 양파까지 건드리기

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings

warnings.filterwarnings('ignore')

# =======================================================================
# 1. 공통 학습 데이터 로드
# =======================================================================
train = pd.read_csv('train.csv')
wholesale_train = pd.read_csv('TRAIN_전국도매_2018-2021.csv')

# 2. 타겟 10대 품목 및 매핑
target_items = [
    {'품목명': '건고추', '품종명': '화건', '거래단위': '30 kg', '등급': '상품'},
    {'품목명': '사과', '품종명': '후지', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '감자', '품종명': '감자 수미', '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '배', '품종명': '신고', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '깐마늘(국산)', '품종명': '깐마늘(국산)', '거래단위': '20 kg', '등급': '상품'},
    {'품목명': '무', '품종명': '무', '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '상추', '품종명': '청', '거래단위': '100 g', '등급': '상품'},
    {'품목명': '배추', '품종명': '배추', '거래단위': '10키로망대', '등급': '상'},
    {'품목명': '양파', '품종명': '양파', '거래단위': '1키로', '등급': '상'},
    {'품목명': '대파', '품종명': '대파(일반)', '거래단위': '1키로단', '등급': '상'}
]

# 💡 건고추 반입량을 '마늘'로 대치
wholesale_name_map = {'깐마늘(국산)': '마늘', '건고추': '마늘'}

test_ids = [f'TEST_{i:02d}' for i in range(25)]
all_validation_results = []

print(f"🚀 총 {len(test_ids)}개 테스트 파일 대상 모의고사 시작 (저장 작물 그룹 3까지 모두 적용!)...\n")

# =======================================================================
# 3. 테스트 파일 루프
# =======================================================================
for test_id in test_ids:
    test_num = test_id.split('_')[1]

    try:
        test = pd.read_csv(f'{test_id}.csv')
        wholesale_test = pd.read_csv(f'TEST_전국도매_{test_num}.csv')
    except FileNotFoundError:
        continue

    print(f"🔄 [{test_id}] 예측 진행 중...")

    # 4. 품목별 모델링 및 예측
    for item in target_items:
        name = f"{item['품목명']}_{item['품종명']}"

        # --- 가격 데이터 ---
        cond_tr_p = (train['품목명'] == item['품목명']) & (train['품종명'] == item['품종명']) & \
                    (train['거래단위'] == item['거래단위']) & (train['등급'] == item['등급'])
        df_tr_p = train[cond_tr_p][['시점', '평균가격(원)']]

        cond_te_p = (test['품목명'] == item['품목명']) & (test['품종명'] == item['품종명']) & \
                    (test['거래단위'] == item['거래단위']) & (test['등급'] == item['등급'])
        df_te_p = test[cond_te_p][['시점', '평균가격(원)']]

        if df_te_p.empty or df_te_p[df_te_p['시점'] == 'T'].empty:
            continue

        # T 시점 실제 가격 저장 및 블라인드 처리
        actual_price = df_te_p[df_te_p['시점'] == 'T']['평균가격(원)'].values[0]
        df_te_p.loc[df_te_p['시점'] == 'T', '평균가격(원)'] = np.nan

        # --- 반입량 데이터 ---
        search_name = wholesale_name_map.get(item['품목명'], item['품목명'])
        cond_tr_s = (wholesale_train['품목명'] == search_name)
        df_tr_s = wholesale_train[cond_tr_s].groupby('시점')['총반입량(kg)'].sum().reset_index()

        cond_te_s = (wholesale_test['품목명'] == search_name)
        df_te_s = wholesale_test[cond_te_s].groupby('시점')['총반입량(kg)'].sum().reset_index()

        # --- 데이터 병합 ---
        df_p = pd.concat([df_tr_p, df_te_p], ignore_index=True)
        df_s = pd.concat([df_tr_s, df_te_s], ignore_index=True)
        df_merged = pd.merge(df_p, df_s, on='시점', how='left')

        # 반입량 결측치 0 처리
        df_merged['총반입량(kg)'] = df_merged['총반입량(kg)'].fillna(0)

        # 💡 시차 2시점 (Lag 2) 적용 및 단위 스케일링(/10000)
        df_merged['exog_lag2'] = df_merged['총반입량(kg)'].shift(2).fillna(method='bfill') / 10000

        # --- 베이스라인 모델 학습 ---
        try:
            # 💡 [그룹 1] 배추 (피크/고변동성/관성형)
            if name == '배추_배추':
                model = sm.tsa.UnobservedComponents(
                    endog=df_merged['평균가격(원)'],
                    level='local linear trend',
                    stochastic_level=True,
                    stochastic_slope=False,
                    freq_seasonal=[{'period': 36, 'harmonics': 3}],
                    autoregressive=2,
                    exog=df_merged['exog_lag2']
                )

            # 💡 [그룹 2] 상추, 대파 (신선 단기 채소/강력한 주기성/장기 추세 배제)
            elif name in ['상추_청', '대파_대파(일반)']:
                model = sm.tsa.UnobservedComponents(
                    endog=df_merged['평균가격(원)'],
                    level='local linear trend',
                    stochastic_level=True,
                    stochastic_slope=False, # 기울기 강제 고정
                    seasonal=36,
                    autoregressive=1,
                    exog=df_merged['exog_lag2']
                )

            # 💡 [그룹 3] 무, 감자, 양파 (저장/뿌리 작물/장기 추세 추종형)
            elif name in ['무_무', '감자_감자 수미', '양파_양파']:
                model = sm.tsa.UnobservedComponents(
                    endog=df_merged['평균가격(원)'],
                    level='local linear trend',
                    stochastic_level=True,
                    stochastic_slope=True,  # 장기 추세 변동 허용!
                    seasonal=36,
                    autoregressive=1,
                    exog=df_merged['exog_lag2']
                )

            # 💡 [그룹 4] 나머지 품목 (건고추, 사과, 배, 깐마늘 - 기본 엔진)
            else:
                model = sm.tsa.UnobservedComponents(
                    endog=df_merged['평균가격(원)'],
                    level='local linear trend',
                    seasonal=36,
                    autoregressive=1,
                    exog=df_merged['exog_lag2']
                )

            res = model.fit(disp=False)

            # 예측 및 오차 계산
            smoothed_prices = res.predict()
            predicted_price = max(smoothed_prices.iloc[-1], 0)

            abs_error = abs(actual_price - predicted_price)
            nmae = abs_error / actual_price if actual_price != 0 else 0

            all_validation_results.append({
                'Test_ID': test_id,
                '품목': name,
                'NMAE': nmae
            })

        except Exception as e:
            pass

# =======================================================================
# 5. 성적표 집계 및 출력
# =======================================================================
print("\n" + "="*50)
print(" 🚨 [종합] 배추/상추/대파/무/감자/양파 맞춤 튜닝 반영 NMAE 성적표")
print("="*50)

if all_validation_results:
    result_df = pd.DataFrame(all_validation_results)

    summary_df = result_df.groupby('품목')['NMAE'].mean().reset_index()
    summary_df = summary_df.sort_values(by='NMAE', ascending=False).reset_index(drop=True)
    summary_df['NMAE'] = summary_df['NMAE'].round(4)

    print(summary_df.to_string(index=False))

    print("-" * 50)
    overall_mean_nmae = summary_df['NMAE'].mean()
    print(f"💡 25개 테스트셋 통합 평균 NMAE: {overall_mean_nmae:.4f}")

    if overall_mean_nmae <= 0.09:
        print("🏆 미쳤습니다! 0.08대 진입 축하드립니다!!")
    elif overall_mean_nmae <= 0.12:
        print("🎯 여전히 초고득점 궤도입니다!")
else:
    print("수집된 결과가 없습니다. 에러 로그를 확인해주세요.")

🚀 총 25개 테스트 파일 대상 모의고사 시작 (저장 작물 그룹 3까지 모두 적용!)...

🔄 [TEST_00] 예측 진행 중...
🔄 [TEST_01] 예측 진행 중...
🔄 [TEST_02] 예측 진행 중...
🔄 [TEST_03] 예측 진행 중...
🔄 [TEST_04] 예측 진행 중...
🔄 [TEST_05] 예측 진행 중...
🔄 [TEST_06] 예측 진행 중...
🔄 [TEST_07] 예측 진행 중...
🔄 [TEST_08] 예측 진행 중...
🔄 [TEST_09] 예측 진행 중...
🔄 [TEST_10] 예측 진행 중...
🔄 [TEST_11] 예측 진행 중...
🔄 [TEST_12] 예측 진행 중...
🔄 [TEST_13] 예측 진행 중...
🔄 [TEST_14] 예측 진행 중...
🔄 [TEST_15] 예측 진행 중...
🔄 [TEST_16] 예측 진행 중...
🔄 [TEST_17] 예측 진행 중...
🔄 [TEST_18] 예측 진행 중...
🔄 [TEST_19] 예측 진행 중...
🔄 [TEST_20] 예측 진행 중...
🔄 [TEST_21] 예측 진행 중...
🔄 [TEST_22] 예측 진행 중...
🔄 [TEST_23] 예측 진행 중...
🔄 [TEST_24] 예측 진행 중...

 🚨 [종합] 배추/상추/대파/무/감자/양파 맞춤 튜닝 반영 NMAE 성적표
             품목   NMAE
          배추_배추 0.1893
          양파_양파 0.1576
            무_무 0.1341
      대파_대파(일반) 0.1235
           상추_청 0.1026
       감자_감자 수미 0.0943
           배_신고 0.0528
          사과_후지 0.0467
         건고추_화건 0.0248
깐마늘(국산)_깐마늘(국산) 0.0073
--------------------------------------------------
💡 25개 테스트셋 통합 평균 NMAE: 0.

In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings

# 경고 메시지 숨김
warnings.filterwarnings('ignore')

# =======================================================================
# 1. 공통 데이터 및 제출 양식 로드
# =======================================================================
train = pd.read_csv('train.csv')
wholesale_train = pd.read_csv('TRAIN_전국도매_2018-2021.csv')
submission = pd.read_csv('sample_submission.csv')

# 2. 타겟 10대 품목 및 매핑
target_items = [
    {'품목명': '건고추', '품종명': '화건', '거래단위': '30 kg', '등급': '상품'},
    {'품목명': '사과', '품종명': '후지', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '감자', '품종명': '감자 수미', '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '배', '품종명': '신고', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '깐마늘(국산)', '품종명': '깐마늘(국산)', '거래단위': '20 kg', '등급': '상품'},
    {'품목명': '무', '품종명': '무', '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '상추', '품종명': '청', '거래단위': '100 g', '등급': '상품'},
    {'품목명': '배추', '품종명': '배추', '거래단위': '10키로망대', '등급': '상'},
    {'품목명': '양파', '품종명': '양파', '거래단위': '1키로', '등급': '상'},
    {'품목명': '대파', '품종명': '대파(일반)', '거래단위': '1키로단', '등급': '상'}
]

# 💡 [핵심] 건고추 반입량을 '마늘'로 대치
wholesale_name_map = {'깐마늘(국산)': '마늘', '건고추': '마늘'}

# 3. 테스트 ID 추출 (제출 양식 기준)
test_ids = submission['시점'].apply(lambda x: x.split('+')[0]).unique()
result_sub = submission.copy().set_index('시점')

print(f"🚀 총 {len(test_ids)}개 테스트 파일 대상 최종 미래 3순 예측 시작...\n")

# =======================================================================
# 4. 실전 예측 루프
# =======================================================================
for test_id in test_ids:
    test_num = test_id.split('_')[1]

    try:
        test = pd.read_csv(f'{test_id}.csv')
        wholesale_test = pd.read_csv(f'TEST_전국도매_{test_num}.csv')
    except FileNotFoundError:
        print(f"  [경고] {test_id} 관련 파일을 찾을 수 없어 건너뜁니다.")
        continue

    print(f"🔄 [{test_id}] 미래 예측 진행 중...")

    for item in target_items:
        name = f"{item['품목명']}_{item['품종명']}"
        col_name = item['품목명']

        # --- 가격 데이터 ---
        cond_tr_p = (train['품목명'] == item['품목명']) & (train['품종명'] == item['품종명']) & \
                    (train['거래단위'] == item['거래단위']) & (train['등급'] == item['등급'])
        df_tr_p = train[cond_tr_p][['시점', '평균가격(원)']]

        cond_te_p = (test['품목명'] == item['품목명']) & (test['품종명'] == item['품종명']) & \
                    (test['거래단위'] == item['거래단위']) & (test['등급'] == item['등급'])
        df_te_p = test[cond_te_p][['시점', '평균가격(원)']]

        # --- 반입량 데이터 ---
        search_name = wholesale_name_map.get(item['품목명'], item['품목명'])
        cond_tr_s = (wholesale_train['품목명'] == search_name)
        df_tr_s = wholesale_train[cond_tr_s].groupby('시점')['총반입량(kg)'].sum().reset_index()

        cond_te_s = (wholesale_test['품목명'] == search_name)
        df_te_s = wholesale_test[cond_te_s].groupby('시점')['총반입량(kg)'].sum().reset_index()

        # --- 데이터 병합 ---
        df_p = pd.concat([df_tr_p, df_te_p], ignore_index=True)
        df_s = pd.concat([df_tr_s, df_te_s], ignore_index=True)
        df_merged = pd.merge(df_p, df_s, on='시점', how='left')

        # 반입량 결측치 0 처리
        df_merged['총반입량(kg)'] = df_merged['총반입량(kg)'].fillna(0)

        # 💡 시차 2시점 (Lag 2) 훈련용 외생변수 생성 및 스케일링(/10000)
        df_merged['exog_lag2'] = df_merged['총반입량(kg)'].shift(2).fillna(method='bfill') / 10000

        # 💡 [핵심] 미래 3순 예측을 위한 시차 외생변수 (Future Exog) 정밀 계산
        # Lag 2 모델이므로:
        # T+1 시점 예측 -> T-1 시점 물량 사용
        # T+2 시점 예측 -> T 시점 물량 사용
        # T+3 시점 예측 -> T+1 시점 물량 사용 (모르니까 T 시점 유지)
        recent_vols = df_merged['총반입량(kg)'].tail(2).values / 10000
        vol_T_minus_1 = recent_vols[0]
        vol_T = recent_vols[1]
        future_exog = [vol_T_minus_1, vol_T, vol_T]

        # --- 맞춤형 모델 학습 및 예측 ---
        try:
            # 💡 [맞춤형 엔진 1] 배추 (피크/고변동성/관성형)
            if name == '배추_배추':
                model = sm.tsa.UnobservedComponents(
                    endog=df_merged['평균가격(원)'],
                    level='local linear trend',
                    stochastic_level=True,
                    stochastic_slope=False,
                    freq_seasonal=[{'period': 36, 'harmonics': 3}],
                    autoregressive=2,
                    exog=df_merged['exog_lag2']
                )

            # 💡 [맞춤형 엔진 2] 상추, 대파 (신선 단기 채소/강력한 주기성/장기 추세 배제)
            elif name in ['상추_청', '대파_대파(일반)']:
                model = sm.tsa.UnobservedComponents(
                    endog=df_merged['평균가격(원)'],
                    level='local linear trend',
                    stochastic_level=True,
                    stochastic_slope=False,
                    seasonal=36,
                    autoregressive=1,
                    exog=df_merged['exog_lag2']
                )

            # 💡 [맞춤형 엔진 3] 무, 감자, 양파 (저장/뿌리 작물/장기 추세 추종형)
            elif name in ['무_무', '감자_감자 수미', '양파_양파']:
                model = sm.tsa.UnobservedComponents(
                    endog=df_merged['평균가격(원)'],
                    level='local linear trend',
                    stochastic_level=True,
                    stochastic_slope=True,
                    seasonal=36,
                    autoregressive=1,
                    exog=df_merged['exog_lag2']
                )

            # 💡 [기본 엔진] 사과, 배, 마늘, 건고추
            else:
                model = sm.tsa.UnobservedComponents(
                    endog=df_merged['평균가격(원)'],
                    level='local linear trend',
                    seasonal=36,
                    autoregressive=1,
                    exog=df_merged['exog_lag2']
                )

            res = model.fit(disp=False)

            # 💡 미래 3순(T+1, T+2, T+3) 예측 수행
            forecasts = res.forecast(steps=3, exog=future_exog)

            # 하한선: 가격이 0원 미만이 되는 것을 방지
            forecasts = np.maximum(forecasts, 0)

            # 제출 폼에 기록
            result_sub.loc[f'{test_id}+1순', col_name] = forecasts.iloc[0]
            result_sub.loc[f'{test_id}+2순', col_name] = forecasts.iloc[1]
            result_sub.loc[f'{test_id}+3순', col_name] = forecasts.iloc[2]

        except Exception as e:
            # 에러 발생 시 최후의 방어선: 최근 실제 가격으로 동결 대치
            fallback_price = df_merged['평균가격(원)'].dropna().iloc[-1]
            result_sub.loc[f'{test_id}+1순', col_name] = fallback_price
            result_sub.loc[f'{test_id}+2순', col_name] = fallback_price
            result_sub.loc[f'{test_id}+3순', col_name] = fallback_price

# =======================================================================
# 5. 제출 파일 저장
# =======================================================================
result_sub.reset_index(inplace=True)
result_sub.to_csv('final_submission_TopTier.csv', index=False)
print("\n🎉 예측 완료! 'final_submission_TopTier.csv' 파일이 생성되었습니다. 바로 제출해보세요!")

🚀 총 25개 테스트 파일 대상 최종 미래 3순 예측 시작...

🔄 [TEST_00] 미래 예측 진행 중...
🔄 [TEST_01] 미래 예측 진행 중...
🔄 [TEST_02] 미래 예측 진행 중...
🔄 [TEST_03] 미래 예측 진행 중...
🔄 [TEST_04] 미래 예측 진행 중...
🔄 [TEST_05] 미래 예측 진행 중...
🔄 [TEST_06] 미래 예측 진행 중...
🔄 [TEST_07] 미래 예측 진행 중...
🔄 [TEST_08] 미래 예측 진행 중...
🔄 [TEST_09] 미래 예측 진행 중...
🔄 [TEST_10] 미래 예측 진행 중...
🔄 [TEST_11] 미래 예측 진행 중...
🔄 [TEST_12] 미래 예측 진행 중...
🔄 [TEST_13] 미래 예측 진행 중...
🔄 [TEST_14] 미래 예측 진행 중...
🔄 [TEST_15] 미래 예측 진행 중...
🔄 [TEST_16] 미래 예측 진행 중...
🔄 [TEST_17] 미래 예측 진행 중...
🔄 [TEST_18] 미래 예측 진행 중...
🔄 [TEST_19] 미래 예측 진행 중...
🔄 [TEST_20] 미래 예측 진행 중...
🔄 [TEST_21] 미래 예측 진행 중...
🔄 [TEST_22] 미래 예측 진행 중...
🔄 [TEST_23] 미래 예측 진행 중...
🔄 [TEST_24] 미래 예측 진행 중...

🎉 예측 완료! 'final_submission_TopTier.csv' 파일이 생성되었습니다. 바로 제출해보세요!


4그룹도 대치

In [3]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings

# 경고 메시지 숨김
warnings.filterwarnings('ignore')

# =======================================================================
# 1. 공통 데이터 및 제출 양식 로드
# =======================================================================
train = pd.read_csv('train.csv')
wholesale_train = pd.read_csv('TRAIN_전국도매_2018-2021.csv')
submission = pd.read_csv('sample_submission.csv')

# 2. 타겟 10대 품목 및 매핑
target_items = [
    {'품목명': '건고추', '품종명': '화건', '거래단위': '30 kg', '등급': '상품'},
    {'품목명': '사과', '품종명': '후지', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '감자', '품종명': '감자 수미', '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '배', '품종명': '신고', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '깐마늘(국산)', '품종명': '깐마늘(국산)', '거래단위': '20 kg', '등급': '상품'},
    {'품목명': '무', '품종명': '무', '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '상추', '품종명': '청', '거래단위': '100 g', '등급': '상품'},
    {'품목명': '배추', '품종명': '배추', '거래단위': '10키로망대', '등급': '상'},
    {'품목명': '양파', '품종명': '양파', '거래단위': '1키로', '등급': '상'},
    {'품목명': '대파', '품종명': '대파(일반)', '거래단위': '1키로단', '등급': '상'}
]

# 💡 건고추 반입량을 '마늘'로 대치
wholesale_name_map = {'깐마늘(국산)': '마늘', '건고추': '마늘'}

# 3. 테스트 ID 추출 (제출 양식 기준)
test_ids = submission['시점'].apply(lambda x: x.split('+')[0]).unique()
result_sub = submission.copy().set_index('시점')

print(f"🚀 총 {len(test_ids)}개 테스트 파일 대상 최종 미래 3순 예측 시작 (전 품목 맞춤형 엔진 탑재!)...\n")

# =======================================================================
# 4. 실전 예측 루프
# =======================================================================
for test_id in test_ids:
    test_num = test_id.split('_')[1]

    try:
        test = pd.read_csv(f'{test_id}.csv')
        wholesale_test = pd.read_csv(f'TEST_전국도매_{test_num}.csv')
    except FileNotFoundError:
        continue

    print(f"🔄 [{test_id}] 미래 예측 진행 중...")

    for item in target_items:
        name = f"{item['품목명']}_{item['품종명']}"
        col_name = item['품목명']

        # --- 가격 데이터 ---
        cond_tr_p = (train['품목명'] == item['품목명']) & (train['품종명'] == item['품종명']) & \
                    (train['거래단위'] == item['거래단위']) & (train['등급'] == item['등급'])
        df_tr_p = train[cond_tr_p][['시점', '평균가격(원)']]

        cond_te_p = (test['품목명'] == item['품목명']) & (test['품종명'] == item['품종명']) & \
                    (test['거래단위'] == item['거래단위']) & (test['등급'] == item['등급'])
        df_te_p = test[cond_te_p][['시점', '평균가격(원)']]

        # --- 반입량 데이터 ---
        search_name = wholesale_name_map.get(item['품목명'], item['품목명'])
        cond_tr_s = (wholesale_train['품목명'] == search_name)
        df_tr_s = wholesale_train[cond_tr_s].groupby('시점')['총반입량(kg)'].sum().reset_index()

        cond_te_s = (wholesale_test['품목명'] == search_name)
        df_te_s = wholesale_test[cond_te_s].groupby('시점')['총반입량(kg)'].sum().reset_index()

        # --- 데이터 병합 ---
        df_p = pd.concat([df_tr_p, df_te_p], ignore_index=True)
        df_s = pd.concat([df_tr_s, df_te_s], ignore_index=True)
        df_merged = pd.merge(df_p, df_s, on='시점', how='left')

        # 반입량 결측치 0 처리
        df_merged['총반입량(kg)'] = df_merged['총반입량(kg)'].fillna(0)

        # 💡 시차 2시점 (Lag 2) 훈련용 외생변수 생성 및 스케일링(/10000)
        df_merged['exog_lag2'] = df_merged['총반입량(kg)'].shift(2).fillna(method='bfill') / 10000

        # 미래 3순 예측용 물량 계산
        recent_vols = df_merged['총반입량(kg)'].tail(2).values / 10000
        vol_T_minus_1 = recent_vols[0]
        vol_T = recent_vols[1]
        future_exog = [vol_T_minus_1, vol_T, vol_T]

        # --- 맞춤형 모델 학습 및 예측 ---
        try:
            # 💡 [맞춤형 엔진 1] 배추 (피크/고변동성/관성형)
            if name == '배추_배추':
                model = sm.tsa.UnobservedComponents(
                    endog=df_merged['평균가격(원)'],
                    level='local linear trend',
                    stochastic_level=True,
                    stochastic_slope=False,
                    freq_seasonal=[{'period': 36, 'harmonics': 3}],
                    autoregressive=2,
                    exog=df_merged['exog_lag2']
                )

            # 💡 [맞춤형 엔진 2] 상추, 대파 (신선 단기 채소/강력한 주기성/장기 추세 배제)
            elif name in ['상추_청', '대파_대파(일반)']:
                model = sm.tsa.UnobservedComponents(
                    endog=df_merged['평균가격(원)'],
                    level='local linear trend',
                    stochastic_level=True,
                    stochastic_slope=False,
                    seasonal=36,
                    autoregressive=1,
                    exog=df_merged['exog_lag2']
                )

            # 💡 [맞춤형 엔진 3] 무, 감자, 양파 (저장/뿌리 작물/장기 추세 추종형)
            elif name in ['무_무', '감자_감자 수미', '양파_양파']:
                model = sm.tsa.UnobservedComponents(
                    endog=df_merged['평균가격(원)'],
                    level='local linear trend',
                    stochastic_level=True,
                    stochastic_slope=True,
                    seasonal=36,
                    autoregressive=1,
                    exog=df_merged['exog_lag2']
                )

            # 💡 [맞춤형 엔진 4] 사과, 배, 깐마늘, 건고추 (과일/양념류 평탄 유지형)
            else:
                model = sm.tsa.UnobservedComponents(
                    endog=df_merged['평균가격(원)'],
                    level='local linear trend',
                    stochastic_level=True,     # 명절/충격 점프는 허용
                    stochastic_slope=False,    # 🚀 핵심: 하지만 쓸데없는 장기 상승/하락 추세는 방지
                    seasonal=36,
                    autoregressive=1,
                    exog=df_merged['exog_lag2']
                )

            res = model.fit(disp=False)

            # 미래 3순(T+1, T+2, T+3) 예측 수행
            forecasts = res.forecast(steps=3, exog=future_exog)
            forecasts = np.maximum(forecasts, 0) # 음수 방지

            # 제출 폼에 기록
            result_sub.loc[f'{test_id}+1순', col_name] = forecasts.iloc[0]
            result_sub.loc[f'{test_id}+2순', col_name] = forecasts.iloc[1]
            result_sub.loc[f'{test_id}+3순', col_name] = forecasts.iloc[2]

        except Exception as e:
            # 에러 발생 시 직전 가격으로 동결
            fallback_price = df_merged['평균가격(원)'].dropna().iloc[-1]
            result_sub.loc[f'{test_id}+1순', col_name] = fallback_price
            result_sub.loc[f'{test_id}+2순', col_name] = fallback_price
            result_sub.loc[f'{test_id}+3순', col_name] = fallback_price

# =======================================================================
# 5. 제출 파일 저장
# =======================================================================
result_sub.reset_index(inplace=True)
result_sub.to_csv('final_submission_Group4_Tuned.csv', index=False)
print("\n🎉 모든 그룹의 맞춤형 엔진 적용이 완료되었습니다! 'final_submission_Group4_Tuned.csv'로 제출해보세요!")

🚀 총 25개 테스트 파일 대상 최종 미래 3순 예측 시작 (전 품목 맞춤형 엔진 탑재!)...

🔄 [TEST_00] 미래 예측 진행 중...
🔄 [TEST_01] 미래 예측 진행 중...
🔄 [TEST_02] 미래 예측 진행 중...
🔄 [TEST_03] 미래 예측 진행 중...
🔄 [TEST_04] 미래 예측 진행 중...
🔄 [TEST_05] 미래 예측 진행 중...
🔄 [TEST_06] 미래 예측 진행 중...
🔄 [TEST_07] 미래 예측 진행 중...
🔄 [TEST_08] 미래 예측 진행 중...
🔄 [TEST_09] 미래 예측 진행 중...
🔄 [TEST_10] 미래 예측 진행 중...
🔄 [TEST_11] 미래 예측 진행 중...
🔄 [TEST_12] 미래 예측 진행 중...
🔄 [TEST_13] 미래 예측 진행 중...
🔄 [TEST_14] 미래 예측 진행 중...
🔄 [TEST_15] 미래 예측 진행 중...
🔄 [TEST_16] 미래 예측 진행 중...
🔄 [TEST_17] 미래 예측 진행 중...
🔄 [TEST_18] 미래 예측 진행 중...
🔄 [TEST_19] 미래 예측 진행 중...
🔄 [TEST_20] 미래 예측 진행 중...
🔄 [TEST_21] 미래 예측 진행 중...
🔄 [TEST_22] 미래 예측 진행 중...
🔄 [TEST_23] 미래 예측 진행 중...
🔄 [TEST_24] 미래 예측 진행 중...

🎉 모든 그룹의 맞춤형 엔진 적용이 완료되었습니다! 'final_submission_Group4_Tuned.csv'로 제출해보세요!


In [4]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings

# 경고 메시지 숨김
warnings.filterwarnings('ignore')

# =======================================================================
# 1. 공통 데이터 및 제출 양식 로드
# =======================================================================
train = pd.read_csv('train.csv')
wholesale_train = pd.read_csv('TRAIN_전국도매_2018-2021.csv')
submission = pd.read_csv('sample_submission.csv')

# 2. 타겟 10대 품목 및 매핑
target_items = [
    {'품목명': '건고추', '품종명': '화건', '거래단위': '30 kg', '등급': '상품'},
    {'품목명': '사과', '품종명': '후지', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '감자', '품종명': '감자 수미', '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '배', '품종명': '신고', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '깐마늘(국산)', '품종명': '깐마늘(국산)', '거래단위': '20 kg', '등급': '상품'},
    {'품목명': '무', '품종명': '무', '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '상추', '품종명': '청', '거래단위': '100 g', '등급': '상품'},
    {'품목명': '배추', '품종명': '배추', '거래단위': '10키로망대', '등급': '상'},
    {'품목명': '양파', '품종명': '양파', '거래단위': '1키로', '등급': '상'},
    {'품목명': '대파', '품종명': '대파(일반)', '거래단위': '1키로단', '등급': '상'}
]

# 💡 건고추 반입량을 '마늘'로 대치
wholesale_name_map = {'깐마늘(국산)': '마늘', '건고추': '마늘'}

# 3. 테스트 ID 추출 및 결과 저장소 준비
test_ids = submission['시점'].apply(lambda x: x.split('+')[0]).unique()
result_sub = submission.copy().set_index('시점')
all_validation_results = [] # NMAE 채점표 저장용

print(f"🚀 총 {len(test_ids)}개 테스트 파일 대상 모의고사 채점 및 파일 생성 시작...\n")

# =======================================================================
# 4. 실전 예측 및 NMAE 검증 루프
# =======================================================================
for test_id in test_ids:
    test_num = test_id.split('_')[1]

    try:
        test = pd.read_csv(f'{test_id}.csv')
        wholesale_test = pd.read_csv(f'TEST_전국도매_{test_num}.csv')
    except FileNotFoundError:
        continue

    print(f"🔄 [{test_id}] 예측 진행 중...")

    for item in target_items:
        name = f"{item['품목명']}_{item['품종명']}"
        col_name = item['품목명']

        # --- 가격 데이터 ---
        cond_tr_p = (train['품목명'] == item['품목명']) & (train['품종명'] == item['품종명']) & \
                    (train['거래단위'] == item['거래단위']) & (train['등급'] == item['등급'])
        df_tr_p = train[cond_tr_p][['시점', '평균가격(원)']]

        cond_te_p = (test['품목명'] == item['품목명']) & (test['품종명'] == item['품종명']) & \
                    (test['거래단위'] == item['거래단위']) & (test['등급'] == item['등급'])
        df_te_p = test[cond_te_p][['시점', '평균가격(원)']]

        # 💡 [NMAE 검증용] T 시점 실제 가격 빼두고 블라인드(NaN) 처리
        actual_price = None
        if not df_te_p.empty and not df_te_p[df_te_p['시점'] == 'T'].empty:
            actual_price = df_te_p[df_te_p['시점'] == 'T']['평균가격(원)'].values[0]
            df_te_p.loc[df_te_p['시점'] == 'T', '평균가격(원)'] = np.nan

        # --- 반입량 데이터 ---
        search_name = wholesale_name_map.get(item['품목명'], item['품목명'])
        cond_tr_s = (wholesale_train['품목명'] == search_name)
        df_tr_s = wholesale_train[cond_tr_s].groupby('시점')['총반입량(kg)'].sum().reset_index()

        cond_te_s = (wholesale_test['품목명'] == search_name)
        df_te_s = wholesale_test[cond_te_s].groupby('시점')['총반입량(kg)'].sum().reset_index()

        # --- 데이터 병합 ---
        df_p = pd.concat([df_tr_p, df_te_p], ignore_index=True)
        df_s = pd.concat([df_tr_s, df_te_s], ignore_index=True)
        df_merged = pd.merge(df_p, df_s, on='시점', how='left')

        # 반입량 결측치 0 처리
        df_merged['총반입량(kg)'] = df_merged['총반입량(kg)'].fillna(0)

        # 💡 시차 2시점 (Lag 2) 훈련용 외생변수 생성 및 스케일링(/10000)
        df_merged['exog_lag2'] = df_merged['총반입량(kg)'].shift(2).fillna(method='bfill') / 10000

        # 미래 3순 예측용 물량 계산
        recent_vols = df_merged['총반입량(kg)'].tail(2).values / 10000
        vol_T_minus_1 = recent_vols[0]
        vol_T = recent_vols[1]
        future_exog = [vol_T_minus_1, vol_T, vol_T]

        # --- 맞춤형 모델 학습 및 예측 ---
        try:
            # 💡 [맞춤형 엔진 1] 배추 (피크/고변동성/관성형)
            if name == '배추_배추':
                model = sm.tsa.UnobservedComponents(
                    endog=df_merged['평균가격(원)'],
                    level='local linear trend',
                    stochastic_level=True,
                    stochastic_slope=False,
                    freq_seasonal=[{'period': 36, 'harmonics': 3}],
                    autoregressive=2,
                    exog=df_merged['exog_lag2']
                )

            # 💡 [맞춤형 엔진 2] 상추, 대파 (신선 단기 채소/강력한 주기성/장기 추세 배제)
            elif name in ['상추_청', '대파_대파(일반)']:
                model = sm.tsa.UnobservedComponents(
                    endog=df_merged['평균가격(원)'],
                    level='local linear trend',
                    stochastic_level=True,
                    stochastic_slope=False,
                    seasonal=36,
                    autoregressive=1,
                    exog=df_merged['exog_lag2']
                )

            # 💡 [맞춤형 엔진 3] 무, 감자, 양파 (저장/뿌리 작물/장기 추세 추종형)
            elif name in ['무_무', '감자_감자 수미', '양파_양파']:
                model = sm.tsa.UnobservedComponents(
                    endog=df_merged['평균가격(원)'],
                    level='local linear trend',
                    stochastic_level=True,
                    stochastic_slope=True,
                    seasonal=36,
                    autoregressive=1,
                    exog=df_merged['exog_lag2']
                )

            # 💡 [맞춤형 엔진 4] 사과, 배, 깐마늘, 건고추 (과일/양념류 평탄 유지형)
            else:
                model = sm.tsa.UnobservedComponents(
                    endog=df_merged['평균가격(원)'],
                    level='local linear trend',
                    stochastic_level=True,     # 명절/충격 점프는 허용
                    stochastic_slope=False,    # 🚀 핵심: 장기 상승/하락 추세 방지
                    seasonal=36,
                    autoregressive=1,
                    exog=df_merged['exog_lag2']
                )

            res = model.fit(disp=False)

            # 💡 [1] T시점 NMAE 채점
            if actual_price is not None:
                smoothed_prices = res.predict()
                predicted_price_T = max(smoothed_prices.iloc[-1], 0)
                abs_error = abs(actual_price - predicted_price_T)
                nmae = abs_error / actual_price if actual_price != 0 else 0
                all_validation_results.append({
                    'Test_ID': test_id,
                    '품목': name,
                    'NMAE': nmae
                })

            # 💡 [2] 미래 3순(T+1, T+2, T+3) 예측 및 기록 (제출용)
            forecasts = res.forecast(steps=3, exog=future_exog)
            forecasts = np.maximum(forecasts, 0) # 음수 방지

            result_sub.loc[f'{test_id}+1순', col_name] = forecasts.iloc[0]
            result_sub.loc[f'{test_id}+2순', col_name] = forecasts.iloc[1]
            result_sub.loc[f'{test_id}+3순', col_name] = forecasts.iloc[2]

        except Exception as e:
            fallback_price = df_merged['평균가격(원)'].dropna().iloc[-1] if not df_merged['평균가격(원)'].dropna().empty else 0
            result_sub.loc[f'{test_id}+1순', col_name] = fallback_price
            result_sub.loc[f'{test_id}+2순', col_name] = fallback_price
            result_sub.loc[f'{test_id}+3순', col_name] = fallback_price

# =======================================================================
# 5. 성적표 집계 및 출력
# =======================================================================
print("\n" + "="*50)
print(" 🚨 [종합] 그룹 4 튜닝 반영 NMAE 성적표")
print("="*50)

if all_validation_results:
    result_df = pd.DataFrame(all_validation_results)
    summary_df = result_df.groupby('품목')['NMAE'].mean().reset_index()
    summary_df = summary_df.sort_values(by='NMAE', ascending=False).reset_index(drop=True)
    summary_df['NMAE'] = summary_df['NMAE'].round(4)

    print(summary_df.to_string(index=False))
    print("-" * 50)

    overall_mean_nmae = summary_df['NMAE'].mean()
    print(f"💡 25개 테스트셋 통합 평균 NMAE: {overall_mean_nmae:.4f}")

# =======================================================================
# 6. 제출 파일 저장
# =======================================================================
result_sub.reset_index(inplace=True)
result_sub.to_csv('final_submission_Group4_Tuned.csv', index=False)
print("\n🎉 예측 완료! 'final_submission_Group4_Tuned.csv' 파일이 성공적으로 저장되었습니다.")

🚀 총 25개 테스트 파일 대상 모의고사 채점 및 파일 생성 시작...

🔄 [TEST_00] 예측 진행 중...
🔄 [TEST_01] 예측 진행 중...
🔄 [TEST_02] 예측 진행 중...
🔄 [TEST_03] 예측 진행 중...
🔄 [TEST_04] 예측 진행 중...
🔄 [TEST_05] 예측 진행 중...
🔄 [TEST_06] 예측 진행 중...
🔄 [TEST_07] 예측 진행 중...
🔄 [TEST_08] 예측 진행 중...
🔄 [TEST_09] 예측 진행 중...
🔄 [TEST_10] 예측 진행 중...
🔄 [TEST_11] 예측 진행 중...
🔄 [TEST_12] 예측 진행 중...
🔄 [TEST_13] 예측 진행 중...
🔄 [TEST_14] 예측 진행 중...
🔄 [TEST_15] 예측 진행 중...
🔄 [TEST_16] 예측 진행 중...
🔄 [TEST_17] 예측 진행 중...
🔄 [TEST_18] 예측 진행 중...
🔄 [TEST_19] 예측 진행 중...
🔄 [TEST_20] 예측 진행 중...
🔄 [TEST_21] 예측 진행 중...
🔄 [TEST_22] 예측 진행 중...
🔄 [TEST_23] 예측 진행 중...
🔄 [TEST_24] 예측 진행 중...

 🚨 [종합] 그룹 4 튜닝 반영 NMAE 성적표
             품목   NMAE
          배추_배추 0.1893
          양파_양파 0.1576
            무_무 0.1341
      대파_대파(일반) 0.1235
           상추_청 0.1026
       감자_감자 수미 0.0943
           배_신고 0.0528
          사과_후지 0.0467
         건고추_화건 0.0248
깐마늘(국산)_깐마늘(국산) 0.0073
--------------------------------------------------
💡 25개 테스트셋 통합 평균 NMAE: 0.0933

🎉 예측 완료! 'final_subm

최적화



In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings

# 경고 메시지 숨김
warnings.filterwarnings('ignore')

# =======================================================================
# 1. 공통 데이터 및 제출 양식 로드
# =======================================================================
train = pd.read_csv('train.csv')
wholesale_train = pd.read_csv('TRAIN_전국도매_2018-2021.csv')
submission = pd.read_csv('sample_submission.csv')

# 2. 타겟 10대 품목 및 매핑
target_items = [
    {'품목명': '건고추', '품종명': '화건', '거래단위': '30 kg', '등급': '상품'},
    {'품목명': '사과', '품종명': '후지', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '감자', '품종명': '감자 수미', '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '배', '품종명': '신고', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '깐마늘(국산)', '품종명': '깐마늘(국산)', '거래단위': '20 kg', '등급': '상품'},
    {'품목명': '무', '품종명': '무', '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '상추', '품종명': '청', '거래단위': '100 g', '등급': '상품'},
    {'품목명': '배추', '품종명': '배추', '거래단위': '10키로망대', '등급': '상'},
    {'품목명': '양파', '품종명': '양파', '거래단위': '1키로', '등급': '상'},
    {'품목명': '대파', '품종명': '대파(일반)', '거래단위': '1키로단', '등급': '상'}
]

wholesale_name_map = {'깐마늘(국산)': '마늘', '건고추': '마늘'}
test_ids = submission['시점'].apply(lambda x: x.split('+')[0]).unique()
result_sub = submission.copy().set_index('시점')

all_validation_results = []

print("🚀 [1:1 현미경 튜닝 모드] 10대 작물 독립 엔진 가동 시작...\n")

# =======================================================================
# 3. 실전 예측 및 NMAE 검증 루프
# =======================================================================
for test_id in test_ids:
    test_num = test_id.split('_')[1]

    try:
        test = pd.read_csv(f'{test_id}.csv')
        wholesale_test = pd.read_csv(f'TEST_전국도매_{test_num}.csv')
    except FileNotFoundError:
        continue

    print(f"🔄 [{test_id}] 분석 중...")

    for item in target_items:
        name = f"{item['품목명']}_{item['품종명']}"
        col_name = item['품목명']

        # --- 데이터 준비 ---
        df_tr_p = train[(train['품목명'] == item['품목명']) & (train['품종명'] == item['품종명'])][['시점', '평균가격(원)']]
        df_te_p = test[(test['품목명'] == item['품목명']) & (test['품종명'] == item['품종명'])][['시점', '평균가격(원)']]

        actual_price = None
        if not df_te_p.empty and not df_te_p[df_te_p['시점'] == 'T'].empty:
            actual_price = df_te_p[df_te_p['시점'] == 'T']['평균가격(원)'].values[0]
            df_te_p.loc[df_te_p['시점'] == 'T', '평균가격(원)'] = np.nan # 블라인드 처리

        df_p = pd.concat([df_tr_p, df_te_p], ignore_index=True)

        # --- 반입량 데이터 및 Lag 2 적용 ---
        search_name = wholesale_name_map.get(item['품목명'], item['품목명'])
        df_tr_s = wholesale_train[wholesale_train['품목명'] == search_name].groupby('시점')['총반입량(kg)'].sum().reset_index()
        df_te_s = wholesale_test[wholesale_test['품목명'] == search_name].groupby('시점')['총반입량(kg)'].sum().reset_index()
        df_s = pd.concat([df_tr_s, df_te_s], ignore_index=True)

        df_merged = pd.merge(df_p, df_s, on='시점', how='left')
        df_merged['총반입량(kg)'] = df_merged['총반입량(kg)'].fillna(0)

        raw_vols = df_merged['총반입량(kg)'].values / 10000
        df_merged['exog_lag2'] = df_merged['총반입량(kg)'].shift(2).fillna(method='bfill') / 10000

        exog_data = df_merged['exog_lag2']
        future_exog = [raw_vols[-2], raw_vols[-1], raw_vols[-1]]

        # =======================================================================
        # 🎛️ [작물별 커스텀 튜닝 보드] 각 작물의 생리적 특성에 맞춘 파라미터 적용
        # =======================================================================
        try:
            # 🚨 1. 배추 (목표 0.18 -> 0.10): 날카로운 계절성 + 극강의 관성(AR=3)
            if col_name == '배추':
                model = sm.tsa.UnobservedComponents(
                    df_merged['평균가격(원)'], level='local linear trend', stochastic_level=True,
                    stochastic_slope=False, seasonal=36, autoregressive=3, exog=exog_data
                )

            # 🚨 2. 양파 (목표 0.15 -> 0.10): 추세 억제 + 창고 방출 관성(AR=2)
            elif col_name == '양파':
                model = sm.tsa.UnobservedComponents(
                    df_merged['평균가격(원)'], level='local linear trend', stochastic_level=True,
                    stochastic_slope=False, seasonal=36, autoregressive=2, exog=exog_data
                )

            # 🚨 3. 무 (목표 0.13 -> 0.10): 추세 허용 + 단기 관성 연장(AR=2)
            elif col_name == '무':
                model = sm.tsa.UnobservedComponents(
                    df_merged['평균가격(원)'], level='local linear trend', stochastic_level=True,
                    stochastic_slope=True, seasonal=36, autoregressive=2, exog=exog_data
                )

            # 🚨 4. 대파 (목표 0.12 -> 0.10): 날카로운 계절성 제거 + 부드러운 파동(Harmonics=2)
            elif col_name == '대파':
                model = sm.tsa.UnobservedComponents(
                    df_merged['평균가격(원)'], level='local linear trend', stochastic_level=True,
                    stochastic_slope=False, freq_seasonal=[{'period': 36, 'harmonics': 2}], autoregressive=1, exog=exog_data
                )

            # 🥬 5. 상추 (추세 억제형)
            elif col_name == '상추':
                model = sm.tsa.UnobservedComponents(
                    df_merged['평균가격(원)'], level='local linear trend', stochastic_level=True,
                    stochastic_slope=False, seasonal=36, autoregressive=1, exog=exog_data
                )

            # 🥔 6. 감자 (추세 추종형)
            elif col_name == '감자':
                model = sm.tsa.UnobservedComponents(
                    df_merged['평균가격(원)'], level='local linear trend', stochastic_level=True,
                    stochastic_slope=True, seasonal=36, autoregressive=1, exog=exog_data
                )

            # 🍎 7. 사과, 배, 건고추, 마늘 (레벨 점프 억제형 / 안정적 추세)
            else:
                model = sm.tsa.UnobservedComponents(
                    df_merged['평균가격(원)'], level='local linear trend', stochastic_level=False,
                    stochastic_slope=True, seasonal=36, autoregressive=1, exog=exog_data
                )

            # 모델 학습 및 예측
            res = model.fit(disp=False)

            # [T시점 NMAE 채점]
            if actual_price is not None:
                pred_T = max(res.predict().iloc[-1], 0)
                all_validation_results.append({
                    '품목': name,
                    'NMAE': abs(actual_price - pred_T) / actual_price if actual_price != 0 else 0
                })

            # [미래 3순 예측 기록]
            forecasts = np.maximum(res.forecast(steps=3, exog=future_exog), 0)
            result_sub.loc[f'{test_id}+1순', col_name] = forecasts.iloc[0]
            result_sub.loc[f'{test_id}+2순', col_name] = forecasts.iloc[1]
            result_sub.loc[f'{test_id}+3순', col_name] = forecasts.iloc[2]

        except Exception as e:
            # 에러 방어선: 직전 정상 가격으로 대치
            fallback_price = df_merged['평균가격(원)'].dropna().iloc[-1] if not df_merged['평균가격(원)'].dropna().empty else 0
            result_sub.loc[f'{test_id}+1순', col_name] = fallback_price
            result_sub.loc[f'{test_id}+2순', col_name] = fallback_price
            result_sub.loc[f'{test_id}+3순', col_name] = fallback_price

# =======================================================================
# 4. 성적표 집계 및 출력
# =======================================================================
if all_validation_results:
    summary_df = pd.DataFrame(all_validation_results).groupby('품목')['NMAE'].mean().reset_index()
    summary_df = summary_df.sort_values(by='NMAE', ascending=False).reset_index(drop=True)

    print("\n" + "="*50)
    print(" 🚨 [종합] 1:1 현미경 튜닝 대시보드 NMAE 성적표")
    print("="*50)
    print(summary_df.to_string(index=False))
    print(f"\n💡 25개 테스트셋 통합 평균 NMAE: {summary_df['NMAE'].mean():.4f}")

# 제출 파일 저장
result_sub.reset_index().to_csv('final_submission_MicroTuning.csv', index=False)
print("\n🎉 예측 완료! 'final_submission_MicroTuning.csv' 파일이 저장되었습니다.")

🚀 [1:1 현미경 튜닝 모드] 10대 작물 독립 엔진 가동 시작...

🔄 [TEST_00] 분석 중...
🔄 [TEST_01] 분석 중...
🔄 [TEST_02] 분석 중...
🔄 [TEST_03] 분석 중...
🔄 [TEST_04] 분석 중...
🔄 [TEST_05] 분석 중...
🔄 [TEST_06] 분석 중...
🔄 [TEST_07] 분석 중...
🔄 [TEST_08] 분석 중...
🔄 [TEST_09] 분석 중...
🔄 [TEST_10] 분석 중...
🔄 [TEST_11] 분석 중...
🔄 [TEST_12] 분석 중...
🔄 [TEST_13] 분석 중...
🔄 [TEST_14] 분석 중...
🔄 [TEST_15] 분석 중...
🔄 [TEST_16] 분석 중...
🔄 [TEST_17] 분석 중...


In [ ]:
!pip install pytorch_lightning

In [ ]:
! pip install pytorch_forecasting

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.8/399.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.8/159.8 kB 10.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer, QuantileLoss
from pytorch_forecasting.data import GroupNormalizer
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. 데이터 로드 및 기본 전처리
# ==========================================
print("=== 1. 데이터 로드 및 전처리 시작 ===")
train = pd.read_csv('train.csv')
wholesale = pd.read_csv('TRAIN_전국도매_2018-2021.csv')
submission = pd.read_csv('sample_submission.csv')

# 도매 데이터 필터링 및 매핑
wholesale = wholesale[wholesale['시장명'] == '*전국도매시장']
wholesale_name_map = {'깐마늘(국산)': '마늘', '건고추': '고추'}

# 타겟 10대 품목 정보
target_items = [
    {'품목명': '건고추', '품종명': '화건', '등급': '상품'},
    {'품목명': '사과', '품종명': '후지', '등급': '상품'},
    {'품목명': '감자', '품종명': '감자 수미', '등급': '상'},
    {'품목명': '배', '품종명': '신고', '등급': '상품'},
    {'품목명': '깐마늘(국산)', '품종명': '깐마늘(국산)', '등급': '상품'},
    {'품목명': '무', '품종명': '무', '등급': '상'},
    {'품목명': '상추', '품종명': '청', '등급': '상품'},
    {'품목명': '배추', '품종명': '배추', '등급': '상'},
    {'품목명': '양파', '품종명': '양파', '등급': '상'},
    {'품목명': '대파', '품종명': '대파(일반)', '등급': '상'}
]

# 품목별 맞춤 Lag 딕셔너리
lag_dict = {
    '건고추': None, '사과': 4, '감자': 2, '배': 3, '깐마늘(국산)': 3,
    '무': 4, '상추': 2, '배추': 2, '양파': 2, '대파': 2
}

# 시점 데이터를 time_idx, month, soon으로 변환하는 함수
def parse_time(df):
    # '201801상순' -> year: 2018, month: 1, soon: 1(상), 2(중), 3(하)
    df['year'] = df['시점'].str[:4].astype(int)
    df['month'] = df['시점'].str[4:6].astype(int)
    soon_map = {'상순': 1, '중순': 2, '하순': 3}
    df['soon'] = df['시점'].str[6:].map(soon_map).astype(int)

    # 고유한 time_idx 생성 (0, 1, 2, ...)
    unique_times = sorted(df['시점'].unique())
    time_map = {t: i for i, t in enumerate(unique_times)}
    df['time_idx'] = df['시점'].map(time_map)
    return df

train = parse_time(train)
all_time_df = train[['시점', 'time_idx', 'year', 'month', 'soon']].drop_duplicates().sort_values('time_idx')

# ==========================================
# 2. 통합 데이터프레임 구성 (Feature Engineering)
# ==========================================
processed_dfs = []

for item_info in target_items:
    item = item_info['품목명']

    # 1) 가격 데이터
    cond_p = (train['품목명'] == item) & (train['품종명'] == item_info['품종명']) & (train['등급'] == item_info['등급'])
    df_p = train[cond_p][['시점', '평균가격(원)']]

    # 2) 도매 반입량 및 Spread 데이터
    search_name = wholesale_name_map.get(item, item)
    cond_s = (wholesale['품목명'] == search_name)

    try:
        df_s = wholesale[cond_s].groupby('시점').agg(
            총반입량=('총반입량(kg)', 'sum'),
            최고가=('최대거래가격(원/kg)', 'max'),
            최저가=('최소거래가격(원/kg)', 'min'),
            평균가=('평균거래가격(원/kg)', 'mean')
        ).reset_index()
        df_s['Relative_Spread'] = (df_s['최고가'] - df_s['최저가']) / df_s['평균가']
        df_s['Relative_Spread'] = df_s['Relative_Spread'].replace([np.inf, -np.inf], np.nan).fillna(0)
    except KeyError:
        df_s = wholesale[cond_s].groupby('시점')['총반입량(kg)'].sum().reset_index()
        df_s.rename(columns={'총반입량(kg)': '총반입량'}, inplace=True)
        df_s['Relative_Spread'] = 0.0

    # 3) 병합 및 결측치 보간 (선형 보간)
    df_merged = pd.merge(all_time_df, df_p, on='시점', how='left')
    df_merged = pd.merge(df_merged, df_s[['시점', '총반입량', 'Relative_Spread']], on='시점', how='left')

    df_merged['평균가격(원)'] = df_merged['평균가격(원)'].interpolate(method='linear').bfill().ffill()
    df_merged['총반입량'] = df_merged['총반입량'].fillna(0)
    df_merged['Relative_Spread'] = df_merged['Relative_Spread'].fillna(0)

    # 4) 맞춤 Lag 적용 (Shift)
    lag = lag_dict[item]
    if lag is not None:
        df_merged['총반입량_lagged'] = df_merged['총반입량'].shift(lag).fillna(method='bfill')
        df_merged['Spread_lagged'] = df_merged['Relative_Spread'].shift(lag).fillna(method='bfill')
    else:
        # 건고추 등 외생변수가 없는 경우 0으로 처리 (TFT Global Model 구조 통일)
        df_merged['총반입량_lagged'] = 0.0
        df_merged['Spread_lagged'] = 0.0

    # Static Categorical 변수 추가
    df_merged['품목명'] = item
    df_merged['품종명'] = item_info['품종명']
    df_merged['등급'] = item_info['등급']

    processed_dfs.append(df_merged)

final_train_df = pd.concat(processed_dfs, ignore_index=True)

# Categorical 타입 변환 (TFT 요구사항)
for col in ['품목명', '품종명', '등급']:
    final_train_df[col] = final_train_df[col].astype(str).astype('category')

# ==========================================
# 3. TFT Dataset 및 DataLoader 생성
# ==========================================
print("=== 3. TFT 모델 데이터셋 생성 중 ===")
max_prediction_length = 3
max_encoder_length = 36
training_cutoff = final_train_df["time_idx"].max() - max_prediction_length

# 검증 전략: 마지막 3순(T+1~3)을 Validation으로 분리하여 데이터 누수 방지
training_dataset = TimeSeriesDataSet(
    final_train_df[lambda x: x.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="평균가격(원)",
    group_ids=["품목명"],
    min_encoder_length=max_encoder_length // 2,
    max_encoder_length=max_encoder_length,
    min_prediction_length=1,
    max_prediction_length=max_prediction_length,
    static_categoricals=["품목명", "품종명", "등급"],
    time_varying_known_reals=["time_idx", "month", "soon"], # 미래를 아는 변수
    time_varying_unknown_reals=["평균가격(원)", "총반입량_lagged", "Spread_lagged"], # 과거만 아는 변수
    target_normalizer=GroupNormalizer(groups=["품목명"], transformation="softplus"), # 부드러운 양수 정규화
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True
)

validation_dataset = TimeSeriesDataSet.from_dataset(training_dataset, final_train_df, predict=True, stop_randomization=True)

batch_size = 64
train_dataloader = training_dataset.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
val_dataloader = validation_dataset.to_dataloader(train=False, batch_size=batch_size * 2, num_workers=0)

# ==========================================
# 4. TFT 모델 선언 및 학습 (Training Optimization)
# ==========================================
print("=== 4. TFT 모델 학습 시작 ===")
pl.seed_everything(42)

early_stop_callback = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=5, verbose=False, mode="min")
lr_logger = LearningRateMonitor()

trainer = pl.Trainer(
    max_epochs=30,
    accelerator="auto",
    enable_model_summary=True,
    gradient_clip_val=0.1,
    callbacks=[lr_logger, early_stop_callback],
)

tft = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate=0.03,
    hidden_size=16,
    attention_head_size=2,
    dropout=0.1,
    hidden_continuous_size=8,
    loss=QuantileLoss([0.1, 0.5, 0.9]), # NMAE 최적화를 위한 분위수 손실
    reduce_on_plateau_patience=3,       # ReduceLROnPlateau 적용
)

trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)

print("✅ 모델 학습 완료!")

# ==========================================
# 5. 추론 및 제출 파일 생성 (Inference)
# ==========================================
print("=== 5. 미래 T+1 ~ T+3 예측 및 제출 파일 저장 ===")
# (실제 대회에서는 TEST 데이터셋을 순회하며 context를 구성해야 하지만,
# 여기서는 훈련된 모델 사용법과 양식을 맞추는 핵심 로직을 보여드립니다.)

# 가장 마지막 36개 시점을 Context로 사용하여 미래 3단계를 예측합니다.
best_model_path = trainer.checkpoint_callback.best_model_path
best_tft = TemporalFusionTransformer.load_from_checkpoint(best_model_path)

# 10개 품목에 대해 최종 예측 수행
predictions = best_tft.predict(val_dataloader, mode="quantiles", return_x=False)

# predictions 구조: [batch_size(품목수), max_prediction_length(3), num_quantiles(3)]
# q=0.5 (중앙값)은 인덱스 1에 위치합니다.
median_preds = predictions[:, :, 1].numpy()

# 음수 방지 처리
median_preds = np.maximum(0, median_preds)

# 결과 매핑 (예시로 가장 마지막 TEST_ID에 일괄 적용하는 형태입니다.
# 실제 제출 시 test_ids 루프를 돌며 각 테스트 폴더의 데이터를 context로 넣어주어야 합니다.)
result_sub = submission.copy()
result_sub.set_index('시점', inplace=True)

test_ids = submission['시점'].apply(lambda x: x.split('+')[0]).unique()

# 예시 매핑 (품목 순서는 val_dataloader의 그룹 인덱스와 동일)
item_order = list(validation_dataset.decoded_index['품목명'])

for test_id in test_ids:
    for i, item_name in enumerate(item_order):
        result_sub.loc[f'{test_id}+1순', item_name] = median_preds[i, 0]
        result_sub.loc[f'{test_id}+2순', item_name] = median_preds[i, 1]
        result_sub.loc[f'{test_id}+3순', item_name] = median_preds[i, 2]

result_sub.reset_index(inplace=True)
result_sub.to_csv('final_tft_submission.csv', index=False)
print("🎉 성공적으로 'final_tft_submission.csv' 파일이 저장되었습니다!")

=== 1. 데이터 로드 및 전처리 시작 ===


INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42
INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automati

=== 3. TFT 모델 데이터셋 생성 중 ===
=== 4. TFT 모델 학습 시작 ===


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │    122 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    160 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │  2.1 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │  4.4 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │  2.4 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │    544 │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     32 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  1.4 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │    808 │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │    576 │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │    576 │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │     51 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 22.9 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 22.9 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 378                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

✅ 모델 학습 완료!
=== 5. 미래 T+1 ~ T+3 예측 및 제출 파일 저장 ===


INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch.utilities

🎉 성공적으로 'final_tft_submission.csv' 파일이 저장되었습니다!


In [ ]:
!pip install lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 45.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
import warnings

# [긴급 수리 반영] 최신 패키지명 사용
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer, QuantileLoss
from pytorch_forecasting.data import GroupNormalizer
from sklearn.preprocessing import StandardScaler # 💡 스케일러 추가

warnings.filterwarnings('ignore')

# ==========================================
# 1. 데이터 로드 및 기본 전처리
# ==========================================
print("=== 1. 데이터 로드 및 전처리 시작 ===")
train = pd.read_csv('train.csv')
wholesale = pd.read_csv('TRAIN_전국도매_2018-2021.csv')
submission = pd.read_csv('sample_submission.csv')

wholesale = wholesale[wholesale['시장명'] == '*전국도매시장']
wholesale_name_map = {'깐마늘(국산)': '마늘', '건고추': '고추'}

target_items = [
    {'품목명': '건고추', '품종명': '화건', '등급': '상품'},
    {'품목명': '사과', '품종명': '후지', '등급': '상품'},
    {'품목명': '감자', '품종명': '감자 수미', '등급': '상'},
    {'품목명': '배', '품종명': '신고', '등급': '상품'},
    {'품목명': '깐마늘(국산)', '품종명': '깐마늘(국산)', '등급': '상품'},
    {'품목명': '무', '품종명': '무', '등급': '상'},
    {'품목명': '상추', '품종명': '청', '등급': '상품'},
    {'품목명': '배추', '품종명': '배추', '등급': '상'},
    {'품목명': '양파', '품종명': '양파', '등급': '상'},
    {'품목명': '대파', '품종명': '대파(일반)', '등급': '상'}
]

lag_dict = {
    '건고추': None, '사과': 4, '감자': 2, '배': 3, '깐마늘(국산)': 3,
    '무': 4, '상추': 2, '배추': 2, '양파': 2, '대파': 2
}

def parse_time(df):
    df['year'] = df['시점'].str[:4].astype(int)
    df['month'] = df['시점'].str[4:6].astype(int)
    soon_map = {'상순': 1, '중순': 2, '하순': 3}
    df['soon'] = df['시점'].str[6:].map(soon_map).astype(int)

    unique_times = sorted(df['시점'].unique())
    time_map = {t: i for i, t in enumerate(unique_times)}
    df['time_idx'] = df['시점'].map(time_map)
    return df

train = parse_time(train)
all_time_df = train[['시점', 'time_idx', 'year', 'month', 'soon']].drop_duplicates().sort_values('time_idx')

# ==========================================
# 2. 피처 엔지니어링 및 [로그/스케일링] 적용
# ==========================================
processed_dfs = []

for item_info in target_items:
    item = item_info['품목명']

    cond_p = (train['품목명'] == item) & (train['품종명'] == item_info['품종명']) & (train['등급'] == item_info['등급'])
    df_p = train[cond_p][['시점', '평균가격(원)']]

    search_name = wholesale_name_map.get(item, item)
    cond_s = (wholesale['품목명'] == search_name)

    try:
        df_s = wholesale[cond_s].groupby('시점').agg(
            총반입량=('총반입량(kg)', 'sum'),
            최고가=('최대거래가격(원/kg)', 'max'),
            최저가=('최소거래가격(원/kg)', 'min'),
            평균가=('평균거래가격(원/kg)', 'mean')
        ).reset_index()
        df_s['Relative_Spread'] = (df_s['최고가'] - df_s['최저가']) / df_s['평균가']
        df_s['Relative_Spread'] = df_s['Relative_Spread'].replace([np.inf, -np.inf], np.nan).fillna(0)
    except KeyError:
        df_s = wholesale[cond_s].groupby('시점')['총반입량(kg)'].sum().reset_index()
        df_s.rename(columns={'총반입량(kg)': '총반입량'}, inplace=True)
        df_s['Relative_Spread'] = 0.0

    df_merged = pd.merge(all_time_df, df_p, on='시점', how='left')
    df_merged = pd.merge(df_merged, df_s[['시점', '총반입량', 'Relative_Spread']], on='시점', how='left')

    df_merged['평균가격(원)'] = df_merged['평균가격(원)'].interpolate(method='linear').bfill().ffill()
    df_merged['총반입량'] = df_merged['총반입량'].fillna(0)
    df_merged['Relative_Spread'] = df_merged['Relative_Spread'].fillna(0)

    lag = lag_dict[item]
    if lag is not None:
        df_merged['총반입량_lagged'] = df_merged['총반입량'].shift(lag).fillna(method='bfill')
        df_merged['Spread_lagged'] = df_merged['Relative_Spread'].shift(lag).fillna(method='bfill')
    else:
        df_merged['총반입량_lagged'] = 0.0
        df_merged['Spread_lagged'] = 0.0

    df_merged['품목명'] = item
    df_merged['품종명'] = item_info['품종명']
    df_merged['등급'] = item_info['등급']

    processed_dfs.append(df_merged)

final_train_df = pd.concat(processed_dfs, ignore_index=True)

# Categorical 변환
for col in ['품목명', '품종명', '등급']:
    final_train_df[col] = final_train_df[col].astype(str).astype('category')

# 💡 [핵심 처방 1] 로그 변환 (Log Transformation)
# 원본 가격 백업 (나중에 NMAE 점수 계산용)
final_train_df['평균가격(원)_원본'] = final_train_df['평균가격(원)']
# 타겟과 외생변수에 np.log1p 적용 (Spread가 간혹 음수일 수 있으므로 0 이상으로 클리핑 후 적용)
final_train_df['평균가격(원)_log'] = np.log1p(final_train_df['평균가격(원)'])
final_train_df['총반입량_lagged_log'] = np.log1p(final_train_df['총반입량_lagged'])
final_train_df['Spread_lagged_log'] = np.log1p(np.maximum(final_train_df['Spread_lagged'], 0))

# 💡 [핵심 처방 2] 연속형 변수 StandardScaler 적용
scaler = StandardScaler()
continuous_cols = ['총반입량_lagged_log', 'Spread_lagged_log']
final_train_df[continuous_cols] = scaler.fit_transform(final_train_df[continuous_cols])

# ==========================================
# 3. TFT Dataset 생성
# ==========================================
print("=== 3. TFT 모델 데이터셋 생성 중 ===")
max_prediction_length = 3
max_encoder_length = 36
training_cutoff = final_train_df["time_idx"].max() - max_prediction_length

training_dataset = TimeSeriesDataSet(
    final_train_df[lambda x: x.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="평균가격(원)_log", # 타겟을 로그 변환된 값으로 지정!
    group_ids=["품목명"],
    min_encoder_length=max_encoder_length // 2,
    max_encoder_length=max_encoder_length,
    min_prediction_length=1,
    max_prediction_length=max_prediction_length,
    static_categoricals=["품목명", "품종명", "등급"],
    time_varying_known_reals=["time_idx", "month", "soon"],
    time_varying_unknown_reals=["평균가격(원)_log", "총반입량_lagged_log", "Spread_lagged_log"],
    target_normalizer=GroupNormalizer(groups=["품목명"], transformation="relu"), # Log값 음수 예측 방지
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True # 💡 결측 허용 해결책 포함
)

validation_dataset = TimeSeriesDataSet.from_dataset(training_dataset, final_train_df, predict=True, stop_randomization=True)

batch_size = 64
train_dataloader = training_dataset.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
val_dataloader = validation_dataset.to_dataloader(train=False, batch_size=batch_size * 2, num_workers=0)

# ==========================================
# 4. TFT 모델 선언 및 학습 (Model Lightening 적용)
# ==========================================
print("=== 4. TFT 모델 학습 시작 (경량화 적용) ===")
pl.seed_everything(42)

early_stop_callback = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=5, verbose=False, mode="min")
lr_logger = LearningRateMonitor()

trainer = pl.Trainer(
    max_epochs=30,
    accelerator="auto",
    enable_model_summary=True,
    gradient_clip_val=0.1,
    callbacks=[lr_logger, early_stop_callback],
)

tft = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate=0.03,
    hidden_size=16,               # 💡 [핵심 처방 3] 과적합 방지를 위한 모델 경량화
    attention_head_size=2,
    dropout=0.1,
    hidden_continuous_size=8,     # 💡 [핵심 처방 3]
    loss=QuantileLoss([0.1, 0.5, 0.9]),
    reduce_on_plateau_patience=3,
)

trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)

print("✅ 모델 학습 완료!")

# ==========================================
# 5. 검증 데이터 NMAE 평가 및 추론
# ==========================================
print("=== 5. Validation NMAE 계산 및 복원(expm1) ===")
best_model_path = trainer.checkpoint_callback.best_model_path
best_tft = TemporalFusionTransformer.load_from_checkpoint(best_model_path)

# 💡 [해결] 데이터프레임을 자르지 않고, DataLoader에서 정답을 직접 꺼냅니다!
# 이렇게 하면 예측값과 크기(10, 3)가 완벽하게 일치합니다.
actuals_log = torch.cat([y[0] for x, y in iter(val_dataloader)]).numpy()
actuals = np.expm1(actuals_log) # 로그(log) 상태인 정답을 원래 가격으로 복원

# Validation 예측 (Log 스케일 상태의 예측값)
predictions = best_tft.predict(val_dataloader, mode="quantiles", return_x=False)

# q=0.5 (중앙값) 추출
median_preds_log = predictions[:, :, 1].numpy()

# 💡 예측값을 다시 실제 가격 단위로 복원 (Inverse Transform)
median_preds = np.expm1(median_preds_log)
median_preds = np.maximum(0, median_preds) # 최종 음수 방지

# --- Validation NMAE 계산 ---
# 이제 actuals와 median_preds의 크기가 완벽히 동일하므로 에러가 나지 않습니다!
nmae_scores = np.abs(actuals - median_preds) / (actuals + 1e-8) # 0으로 나누기 방지
final_nmae = np.mean(nmae_scores)

print(f"\n🚀 [자체 검증 결과] Validation NMAE Score: {final_nmae:.4f}")
if final_nmae < 0.20:
    print("🔥 목표 달성! 0.1대 점수로 진입했습니다!!")

# --- 제출 파일 매핑 ---
result_sub = submission.copy()
result_sub.set_index('시점', inplace=True)
test_ids = submission['시점'].apply(lambda x: x.split('+')[0]).unique()

# 예측 결과의 품목 순서 가져오기
item_order = list(validation_dataset.decoded_index['품목명'])

for test_id in test_ids:
    for i, item_name in enumerate(item_order):
        result_sub.loc[f'{test_id}+1순', item_name] = median_preds[i, 0]
        result_sub.loc[f'{test_id}+2순', item_name] = median_preds[i, 1]
        result_sub.loc[f'{test_id}+3순', item_name] = median_preds[i, 2]

result_sub.reset_index(inplace=True)
result_sub.to_csv('final_tft_optimized_submission.csv', index=False)
print("🎉 성공적으로 'final_tft_optimized_submission.csv' 파일이 저장되었습니다!")

=== 1. 데이터 로드 및 전처리 시작 ===


INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42
INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automati

=== 3. TFT 모델 데이터셋 생성 중 ===
=== 4. TFT 모델 학습 시작 (경량화 적용) ===


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │    122 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    160 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │  2.1 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │  4.4 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │  2.4 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │    544 │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     32 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  1.4 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │    808 │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │    576 │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │    576 │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │     51 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 22.9 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 22.9 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 378                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

✅ 모델 학습 완료!
=== 5. Validation NMAE 계산 및 복원(expm1) ===


INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch.utilities


🚀 [자체 검증 결과] Validation NMAE Score: 0.1710
🔥 목표 달성! 0.1대 점수로 진입했습니다!!
🎉 성공적으로 'final_tft_optimized_submission.csv' 파일이 저장되었습니다!
